# Scop3P

A comprehensive database of human phosphosites within their full context. Scop3P integrates sequences (UniProtKB/Swiss-Prot), structures (PDB), and uniformly reprocessed phosphoproteomics data (PRIDE) to annotate all known human phosphosites. 

Scop3P, available at https://iomics.ugent.be/scop3p, presents a unique resource for visualization and analysis of phosphosites and for understanding of phosphosite structure–function relationships.

Please cite: https://doi.org/10.1021/acs.jproteome.0c00306

# Scop3P-Structural and biophysical visualization framework

This notebook renders analysis in multiple tabs:


>1. Fetch PTMs using Scop3P API and all single-site UniProt PTM features
>2. Fetch disease variants using UniProt API 
>3. 3D visualization (Mapping PTMs to experimental PDB and AlphaFold structures)
>4. Predict Biophysical properties using Bio2byte tools and map onto 3D structures (single or multi panel)
>5. Residue Interaction Network (RIN) constructions and visualization (We move from 3D to 2.5D to get more insights on local residue interactions)
>6. Structure alignment (Align two structures to see the structural similarity/difference)



In [4]:
import requests, tempfile,json
import pandas as pd 
from b2bTools import SingleSeq, constants
import py3Dmol
import os
import nglview as nv

########
import torch

_original_load = torch.load

def patched_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_load(*args, **kwargs)

torch.load = patched_load



/home/paddy/venvs/ptm/lib/python3.12/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [5]:
import tempfile

state = {
    "acc": None,
    "ptm_results": None,      # keep if you use it
    "ptm_json": None,         # add from state
    "ptm_table": None,
    "variants": None,
    "variants_df": None,      # add from state (or choose one naming)
    "sequence": None,
    "bio2byte_raw": None,
    "dynamic_properties": None,
    "rin_pdb_path": None,
    "af_path": None,
    "rin_html": None,
    "protein_info": None,
    "ptm_source_label": None,
    "workdir": tempfile.mkdtemp(prefix="scop3p_session_")
}


In [6]:
def fetch_protein_modifications(accession):
    """
    Fetch protein modifications from Scop3P for a given UniProt accession.

    Scop3P currently mainly covers human phosphoproteins. For non-human proteins,
    or proteins not present in Scop3P, this function can return None/empty data.
    """
    BASE_URL = "https://iomics.ugent.be/scop3p/api/modifications"
    url = f"{BASE_URL}?accession={accession}"
    headers = {"accept": "application/json"}
    response = requests.get(url, headers=headers, timeout=60)
    if response.status_code == 200:
        try:
            return response.json()
        except ValueError:
            return None          # Scop3P returned a non-JSON body
    return None


_AA1_TO_AA3 = {
    "A": "ALA", "R": "ARG", "N": "ASN", "D": "ASP", "C": "CYS",
    "Q": "GLN", "E": "GLU", "G": "GLY", "H": "HIS", "I": "ILE",
    "L": "LEU", "K": "LYS", "M": "MET", "F": "PHE", "P": "PRO",
    "S": "SER", "T": "THR", "W": "TRP", "Y": "TYR", "V": "VAL",
    "U": "SEC", "O": "PYL",
}


def _residue_from_uniprot_feature(sequence, position, description=""):
    """Return a three-letter residue code for a UniProt PTM feature."""
    desc = (description or "").lower()

    # Prefer explicit residue names in the UniProt PTM description.
    if "phosphoserine" in desc:
        return "SER"
    if "phosphothreonine" in desc:
        return "THR"
    if "phosphotyrosine" in desc:
        return "TYR"

    # Otherwise infer from the sequence position when available.
    try:
        pos = int(position)
        if sequence and 1 <= pos <= len(sequence):
            return _AA1_TO_AA3.get(sequence[pos - 1].upper(), sequence[pos - 1].upper())
    except Exception:
        pass

    return ""


def _format_uniprot_evidence(evidences):
    """Condense UniProt feature evidence into evidence codes and literature references."""
    codes, refs = [], []
    for ev in evidences or []:
        code = ev.get("code")
        if code:
            codes.append(code)
        src = ev.get("source") or {}
        name = src.get("name")
        sid = src.get("id")
        if name and sid:
            refs.append(f"{name}:{sid}")
        elif sid:
            refs.append(str(sid))

    return "; ".join(dict.fromkeys(codes)), "; ".join(dict.fromkeys(refs))


def fetch_uniprot_ptms(accession: str) -> pd.DataFrame:
    """
    Fetch all single-site PTM features from the EBI/UniProt Proteins API.

    This uses categories=PTM without restricting feature types, so it can include
    phosphorylation, acetylation, methylation, glycosylation, lipidation and other
    UniProt PTM annotations when present. Only single-residue features are retained,
    meaning begin == end, because the rest of the app maps PTMs to residue positions.
    The UniProt description is trimmed at the first semicolon, for example
    "Phosphotyrosine; by autocatalysis" becomes "Phosphotyrosine".
    """
    url = f"https://www.ebi.ac.uk/proteins/api/features/{accession}"
    params = [("categories", "PTM")]
    headers = {"Accept": "application/json"}
    r = requests.get(url, headers=headers, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()

    sequence = data.get("sequence", "") or ""
    rows = []
    for feat in data.get("features", []) or []:
        if feat.get("category") != "PTM":
            continue

        begin = feat.get("begin")
        end = feat.get("end")
        if begin is None or end is None or str(begin) != str(end):
            continue

        try:
            position = int(begin)
        except Exception:
            continue

        full_desc = feat.get("description") or feat.get("type") or "PTM"
        clean_name = str(full_desc).split(";", 1)[0].strip()
        evidence, reference = _format_uniprot_evidence(feat.get("evidences"))

        rows.append({
            "ACC_ID": accession,
            "residue": _residue_from_uniprot_feature(sequence, position, clean_name),
            "name": clean_name,
            "evidence": evidence,
            "position": position,
            "source": "UniProt",
            "reference": reference,
            "functionalScore": pd.NA,
            "specificSinglyPhosphorylated": pd.NA,
            "feature_type": feat.get("type"),
        })

    cols = [
        "ACC_ID", "residue", "name", "evidence", "position", "source",
        "reference", "functionalScore", "specificSinglyPhosphorylated", "feature_type",
    ]
    return pd.DataFrame(rows, columns=cols)



# ----------------------------
# UniProt protein summary helpers for Tab 1
# ----------------------------
def _safe_html(value):
    """Small HTML escaping helper without requiring an extra import in the notebook."""
    if value is None:
        return ""
    return (str(value)
            .replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
            .replace('"', "&quot;"))


def _get_uniprot_recommended_name(data):
    pdsc = data.get("proteinDescription") or {}
    rec = pdsc.get("recommendedName") or {}
    full = rec.get("fullName") or {}
    if full.get("value"):
        return full.get("value")

    sub = pdsc.get("submissionNames") or []
    if sub:
        full = (sub[0] or {}).get("fullName") or {}
        if full.get("value"):
            return full.get("value")
    return data.get("uniProtkbId") or data.get("primaryAccession") or ""


def _get_uniprot_gene_names(data):
    genes = []
    for g in data.get("genes", []) or []:
        gn = (g.get("geneName") or {}).get("value")
        if gn:
            genes.append(gn)
    return ", ".join(dict.fromkeys(genes))


def _extract_comment_text(comment):
    texts = []
    for t in comment.get("texts", []) or []:
        val = t.get("value")
        if val:
            texts.append(val)
    return " ".join(texts).strip()


def _get_subcellular_locations(data):
    vals = []
    for c in data.get("comments", []) or []:
        if c.get("commentType") != "SUBCELLULAR LOCATION":
            continue
        for loc in c.get("subcellularLocations", []) or []:
            location = ((loc.get("location") or {}).get("value") or "").strip()
            topology = ((loc.get("topology") or {}).get("value") or "").strip()
            orientation = ((loc.get("orientation") or {}).get("value") or "").strip()
            parts = [p for p in [location, topology, orientation] if p]
            if parts:
                vals.append("; ".join(parts))
        txt = _extract_comment_text(c)
        if txt:
            vals.append(txt)
    return "; ".join(dict.fromkeys(vals))


def _get_function_annotation(data, max_chars=260):
    for c in data.get("comments", []) or []:
        if c.get("commentType") == "FUNCTION":
            txt = _extract_comment_text(c)
            if txt:
                return txt[:max_chars].rstrip() + ("..." if len(txt) > max_chars else "")
    return ""


def fetch_uniprot_protein_info(accession: str) -> dict:
    """Fetch compact UniProtKB protein metadata for the Tab 1 information card."""
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    data = r.json()

    organism = ((data.get("organism") or {}).get("scientificName") or "")
    seq = data.get("sequence") or {}
    keywords = [k.get("name") for k in data.get("keywords", []) or [] if k.get("name")]

    return {
        "accession": data.get("primaryAccession") or accession,
        "entry_name": data.get("uniProtkbId") or "",
        "protein_name": _get_uniprot_recommended_name(data),
        "gene": _get_uniprot_gene_names(data),
        "organism": organism,
        "length": seq.get("length") or len(seq.get("value", "") or ""),
        # UniProt REST commonly returns values such as
        # "UniProtKB reviewed (Swiss-Prot)" or "UniProtKB unreviewed (TrEMBL)".
        # Check for reviewed while explicitly excluding unreviewed.
        "reviewed": ("reviewed" in str(data.get("entryType", "")).lower()
                     and "unreviewed" not in str(data.get("entryType", "")).lower()),
        "subcellular_location": _get_subcellular_locations(data),
        "function": _get_function_annotation(data),
        "keywords": ", ".join(keywords[:8]),
    }


def render_protein_info_html(info: dict, ptm_label: str = "") -> str:
    """Render a compact protein information card for Voila."""
    if not info:
        return "<div style='color:#666;'>Protein information will appear after setting/fetching a UniProt accession.</div>"

    reviewed = "Reviewed" if info.get("reviewed") else "Unreviewed"
    subcell = info.get("subcellular_location") or "Not annotated in UniProt"
    function = info.get("function") or "Not shown"
    keywords = info.get("keywords") or ""
    ptm_line = f"<div><b>PTM source:</b> {_safe_html(ptm_label)}</div>" if ptm_label else ""

    return f"""
    <div style="border:1px solid #d9e2ec; border-radius:8px; padding:10px 12px; background:#f8fbff; margin:6px 0 10px 0;">
      <div style="font-size:15px; margin-bottom:4px;"><b>{_safe_html(info.get('protein_name'))}</b></div>
      <div style="display:grid; grid-template-columns: repeat(2, minmax(220px, 1fr)); gap:4px 18px; font-size:13px;">
        <div><b>Accession:</b> {_safe_html(info.get('accession'))} {_safe_html(info.get('entry_name'))}</div>
        <div><b>Gene:</b> {_safe_html(info.get('gene') or 'N/A')}</div>
        <div><b>Organism:</b> {_safe_html(info.get('organism') or 'N/A')}</div>
        <div><b>Length:</b> {_safe_html(info.get('length') or 'N/A')} aa | {_safe_html(reviewed)}</div>
        {ptm_line}
        <div><b>Keywords:</b> {_safe_html(keywords or 'N/A')}</div>
      </div>
      <div style="font-size:13px; margin-top:6px;"><b>Subcellular location:</b> {_safe_html(subcell)}</div>
      <div style="font-size:13px; margin-top:4px;"><b>Function:</b> {_safe_html(function)}</div>
    </div>
    """


In [7]:
import re
def get_modification_table(modifications, accession=None, source="Scop3P"):
    """
    Convert Scop3P modification records into the PTM table schema used by the app.
    """
    base_cols = [
        "ACC_ID", "residue", "name", "evidence", "position", "source",
        "reference", "functionalScore", "specificSinglyPhosphorylated", "feature_type",
    ]

    if not modifications:
        return pd.DataFrame(columns=base_cols)

    df = pd.DataFrame(modifications)

    # Keep legacy Scop3P columns, but make the function robust if any field is absent.
    for col in ["residue", "name", "evidence", "position", "source", "reference", "functionalScore", "specificSinglyPhosphorylated"]:
        if col not in df.columns:
            df[col] = pd.NA

    if "ACC_ID" not in df.columns:
        df["ACC_ID"] = accession
    if "feature_type" not in df.columns:
        df["feature_type"] = "Scop3P"

    # Do not overwrite a meaningful Scop3P source, but fill blanks.
    df["source"] = df["source"].fillna(source)
    df.loc[df["source"].astype(str).str.strip().eq(""), "source"] = source

    return df[base_cols]




def _join_unique_values(*values, sep="; "):
    """Join non-empty scalar/list values while preserving first-seen order."""
    seen, out = set(), []
    for value in values:
        if value is None or (hasattr(pd, "isna") and not isinstance(value, (list, tuple, set)) and pd.isna(value)):
            continue
        if isinstance(value, (list, tuple, set)):
            parts = value
        else:
            # Preserve comma-separated Scop3P PubMed style and semicolon-separated UniProt style.
            parts = re.split(r"\s*[;,]\s*", str(value))
        for part in parts:
            part = str(part).strip()
            if not part or part.lower() in {"nan", "<na>", "none"}:
                continue
            if part not in seen:
                seen.add(part)
                out.append(part)
    return sep.join(out)


def merge_scop3p_uniprot_ptms(scop3p_tbl, uniprot_tbl):
    """
    Merge Scop3P and UniProt PTMs without duplicating the same residue-position site.

    If a site exists in both sources, the displayed row follows Scop3P terminology
    and Scop3P source fields (for example: name='phosphorylation', evidence='Experimental',
    source='UP', feature_type='Scop3P'). UniProt references/evidence are folded into
    the Scop3P row where useful. UniProt-only PTMs are retained as UniProt rows.
    """
    base_cols = [
        "ACC_ID", "residue", "name", "evidence", "position", "source",
        "reference", "functionalScore", "specificSinglyPhosphorylated", "feature_type",
    ]

    scop3p_tbl = scop3p_tbl.copy() if scop3p_tbl is not None else pd.DataFrame(columns=base_cols)
    uniprot_tbl = uniprot_tbl.copy() if uniprot_tbl is not None else pd.DataFrame(columns=base_cols)

    for df in (scop3p_tbl, uniprot_tbl):
        for col in base_cols:
            if col not in df.columns:
                df[col] = pd.NA
        if not df.empty:
            df["position"] = pd.to_numeric(df["position"], errors="coerce").astype("Int64")
            df["ACC_ID"] = df["ACC_ID"].astype(str).str.strip()
            df["residue"] = df["residue"].astype(str).str.strip().str.upper()

    if scop3p_tbl.empty:
        out = uniprot_tbl[base_cols].copy()
        return out.sort_values(["position", "source", "name"], na_position="last").reset_index(drop=True)
    if uniprot_tbl.empty:
        out = scop3p_tbl[base_cols].copy()
        return out.sort_values(["position", "source", "name"], na_position="last").reset_index(drop=True)

    # Use accession + residue + position as the biological site identity.
    key_cols = ["ACC_ID", "residue", "position"]
    scop3p_tbl["_site_key"] = list(map(tuple, scop3p_tbl[key_cols].astype(str).values))
    uniprot_tbl["_site_key"] = list(map(tuple, uniprot_tbl[key_cols].astype(str).values))

    scop3p_by_key = {k: idx for idx, k in enumerate(scop3p_tbl["_site_key"].tolist())}
    merged_rows = scop3p_tbl.copy()

    # Fold UniProt evidence/reference into matching Scop3P rows, but keep Scop3P naming/source style.
    for _, urow in uniprot_tbl.iterrows():
        key = urow["_site_key"]
        if key not in scop3p_by_key:
            continue
        idx = scop3p_by_key[key]
        sref = merged_rows.at[idx, "reference"]
        uref = urow.get("reference", pd.NA)
        merged_rows.at[idx, "reference"] = _join_unique_values(sref, uref, sep=",")

        # Keep the compact Scop3P evidence label when present; otherwise borrow UniProt evidence.
        sev = merged_rows.at[idx, "evidence"]
        if pd.isna(sev) or str(sev).strip() == "":
            merged_rows.at[idx, "evidence"] = urow.get("evidence", pd.NA)

    uniprot_only = uniprot_tbl[~uniprot_tbl["_site_key"].isin(set(scop3p_by_key.keys()))].copy()
    out = pd.concat([merged_rows, uniprot_only], ignore_index=True)
    out = out.drop(columns=[c for c in ["_site_key"] if c in out.columns])
    out = out[base_cols]
    out = out.drop_duplicates(subset=key_cols + ["name", "source", "feature_type"], keep="first")
    return out.sort_values(["position", "source", "name"], na_position="last").reset_index(drop=True)


In [8]:
import requests
import pandas as pd

def fetch_uniprot_variants_disease(accession: str) -> pd.DataFrame:
    """Fetch UniProt (EBI proteins API) variants with disease association for a UniProt accession."""
    url = f"https://www.ebi.ac.uk/proteins/api/variation/{accession}"
    headers = {"Accept": "application/json"}
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()
    data = r.json()

    rows = []
    for feat in data.get("features", []):
        if feat.get("type") != "VARIANT":
            continue
        for assoc in feat.get("association", []):
            if assoc.get("disease") is not True:
                continue
            begin = feat.get("begin")
            try:
                pos = int(begin) if begin is not None else None
            except Exception:
                pos = None
            rows.append({
                "ACC_ID": accession,
                "position": pos,
                "WT": feat.get("wildType"),
                "MT": feat.get("mutatedType"),
                "consequence": feat.get("consequenceType"),
                "disease_name": assoc.get("name"),
            })

    return pd.DataFrame(rows)


In [9]:
import py3Dmol

def display_local_pdb_3D(modification_table, accession):
    view = py3Dmol.view(width=700, height=500)
    view.addModel(open(accession + '.pdb', 'r').read(), 'pdb')

    view.setStyle({}, {'cartoon': {'color': 'silver'}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.35, 'color': 'white'}, {})

    # --- Color phosphosites 
    for _, row in modification_table.iterrows():
        position = str(row['position'])

        # Normalize residue label to avoid mismatches
        residue = str(row['residue']).strip()  # removes trailing spaces etc.

        if residue == 'TYR':
            color = '#2CA02C'
        elif residue == 'SER':
            color = '#1F77B4'
        elif residue == 'THR':
            color = '#FF7F0E'
        else:
            color = '#7B241C'

        sel = {'resi': position}  # add {'chain': row['chain']} if needed

        view.addStyle(sel, {'stick': {'color': color}})
        view.addStyle(sel, {'sphere': {'color': color, 'radius': 0.9}})

    # --- Hover for ALL amino acids (all atoms) ---
    view.setHoverable(
        {}, True,
        """
        function(atom, viewer, event, container) {
            if(!atom.label) {
                atom.label = viewer.addLabel(
                    atom.resn + " " + atom.resi + (atom.chain ? (" : " + atom.chain) : ""),
                    {position: atom, backgroundColor: 'mintcream', fontColor: 'black'}
                );
            }
        }
        """,
        """
        function(atom, viewer) {
            if(atom.label) {
                viewer.removeLabel(atom.label);
                delete atom.label;
            }
        }
        """
    )

    view.zoomTo()
    view.render()
    return view


In [10]:
def display_ptm_3D(modification_table, pdb_id, chain=None):
    view = py3Dmol.view(query=f"pdb:{pdb_id}")

    # Protein context
    view.setStyle({}, {'cartoon': {'color': 'skyblue'}})

    # Global surface (NO hover expected here)
    view.addSurface(py3Dmol.VDW, {'opacity': 0.6, 'color': 'white'}, {})

    # ---- Colored modified residues (ATOMS) ----
    for _, row in modification_table.iterrows():
        position = str(row['position'])
        residue  = str(row['residue']).strip()

        if residue == 'TYR':
            color = '#2CA02C'
        elif residue == 'SER':
            color = '#1F77B4'
        elif residue == 'THR':
            color = '#FF7F0E'
        else:
            color = '#7B241C'

        sel = {'resi': position}
        if chain:
            sel['chain'] = chain

        # ATOMS → hover works
        view.addStyle(sel, {'stick':  {'color': color}})
        view.addStyle(sel, {'sphere': {'color': color, 'radius': 0.9}})

    # ---- Hover for ALL amino acids ----
    view.setHoverable(
        {}, True,
        """
        function(atom, viewer, event, container) {
            if (!atom.label) {
                atom.label = viewer.addLabel(
                    atom.resn + " " + atom.resi + (atom.chain ? (" : " + atom.chain) : ""),
                    {position: atom, backgroundColor: 'mintcream', fontColor: 'black'}
                );
            }
        }
        """,
        """
        function(atom, viewer) {
            if (atom.label) {
                viewer.removeLabel(atom.label);
                delete atom.label;
            }
        }
        """
    )

    view.zoomTo()
    view.render()
    return view


In [11]:
import os
import requests

def download_alphafold_pdb(uniprot_acc: str, outdir: str) -> str:
    """
    Downloads AlphaFold DB PDB for a UniProt accession.
    Returns local file path.
    """
    os.makedirs(outdir, exist_ok=True)
    # AFDB file naming convention
    url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_acc}-F1-model_v6.pdb"
    out_path = os.path.join(outdir, f"AF-{uniprot_acc}-F1-model_v6.pdb")

    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)

    return out_path


In [12]:
def fetch_sequence_aminoacids(accession):
    BASE_URL = f"http://uniprot.org/uniprotkb/{accession}.fasta"
    url = f'{BASE_URL}?accession={accession}'
    response = requests.get(url)
    if response.status_code == 200:
        raw_fasta_sequence = response.content.decode("utf-8")
    else:
        raw_fasta_sequence = ""
    
    lines = raw_fasta_sequence.split('\n')
    protein_id = str(lines[0])
    amino_acids = "".join([str(l) for l in lines[1:]])
    
    return protein_id, amino_acids

In [13]:
def predict_biophysical_features(accession, sequence):

    with tempfile.NamedTemporaryFile(prefix="seq_", suffix=".fasta", mode="w") as fp:
        fp.write(f">{accession}\n{sequence}\n")
        fp.flush()
        fp.seek(0)
        
        pred = SingleSeq(fp.name).predict(tools=[constants.TOOL_DYNAMINE, constants.TOOL_DISOMINE, constants.TOOL_EFOLDMINE]).get_all_predictions()
    
    return pred


In [14]:
import colorsys


def pseudocolor(minval, maxval,val):
    """ Convert predicted values min.....max in range Green...Yellow..RED 
        The colors correspond to Red and Green in the HSV colorspace
    """
    minval,maxval=float(minval),float(maxval)
    h = (float(maxval-val) / (maxval-minval)) * 120
    r, g, b = colorsys.hsv_to_rgb(h/360, 1., 1.)
    rgb=map(lambda x: int(255 * x), (r, g, b))
    rgb=tuple(rgb)
    rgb='0x%02x%02x%02x' % rgb
    return rgb

In [15]:
def remap(df):
    BDcolor,EFcolor,DOcolor={},{},{}
    seqpos=0
    min_BD,max_BD=min(df.backbone),max(df.backbone)
    min_DO,max_DO=min(df.disoMine),max(df.disoMine)
    min_EF,max_EF=min(df.earlyFolding),max(df.earlyFolding)
    
    for index, row in df.iterrows():
        seqpos+=1
        BDrescol=pseudocolor(min_BD,max_BD,float(row.backbone))
        DOrescol=pseudocolor(min_EF,max_EF,float(row.disoMine))
        EFrescol=pseudocolor(min_EF,max_EF,float(row.earlyFolding))
        BDcolor[seqpos]=BDrescol
        DOcolor[seqpos]=DOrescol
        EFcolor[seqpos]=EFrescol
        
    return BDcolor,EFcolor,DOcolor
        
        

In [16]:
def display_b2b_3D(dynamic_properties, pdb_path: str):
    BDcolor, EFcolor, DOcolor = remap(dynamic_properties)
    modpos = modification_table.position.tolist()

    view = py3Dmol.view(viewergrid=(2,2))
    with open(pdb_path, "r") as f:
        view.addModel(f.read(), "pdb")

    # IMPORTANT: setStyle(selection, style)
    view.setStyle({}, {'cartoon': {'colorscheme': {'prop':'b','gradient':'rwb','min':0.0,'max':100.0}}}, viewer=(0,0))
    view.setStyle({}, {'cartoon': {'colorscheme': {'prop':'resi','map':BDcolor}}}, viewer=(0,1))
    view.setStyle({}, {'cartoon': {'colorscheme': {'prop':'resi','map':DOcolor}}}, viewer=(1,0))
    view.setStyle({}, {'cartoon': {'colorscheme': {'prop':'resi','map':EFcolor}}}, viewer=(1,1))

    # Surface highlight + pickable overlay on mod residues
    for mod in modpos:
        m = str(mod)
        sel = {'resi': m}

        view.addSurface(py3Dmol.VDW, {'opacity': 1.0}, sel, viewer=(0,0))
        view.addSurface(py3Dmol.VDW, {'opacity': 1.0, 'color': BDcolor[mod]}, sel, viewer=(0,1))
        view.addSurface(py3Dmol.VDW, {'opacity': 1.0, 'color': DOcolor[mod]}, sel, viewer=(1,0))
        view.addSurface(py3Dmol.VDW, {'opacity': 1.0, 'color': EFcolor[mod]}, sel, viewer=(1,1))

        # MAKE IT PICKABLE: opacity must be > 0
        for panel in [(0,0), (0,1), (1,0), (1,1)]:
            view.addStyle(sel, {'sphere': {'radius': 0.8, 'opacity': 0.15}}, viewer=panel)
            # optional: stick helps pickability even more
            # view.addStyle(sel, {'stick': {'opacity': 0.15}}, viewer=panel)

    # Background + hover everywhere (per panel)
    for panel in [(0,0), (0,1), (1,0), (1,1)]:
        view.setBackgroundColor('white', viewer=panel)

        view.setHoverable(
            {},  # hover everywhere
            True,
            """
            function(atom, viewer, event, container) {
                if (!atom.label) {
                    atom.label = viewer.addLabel(
                        atom.resn + " " + atom.resi + (atom.chain ? (" : " + atom.chain) : ""),
                        {position: atom, backgroundColor: 'mintcream', fontColor:'black'}
                    );
                }
            }
            """,
            """
            function(atom, viewer) {
                if (atom.label) {
                    viewer.removeLabel(atom.label);
                    delete atom.label;
                }
            }
            """,
            viewer=panel
        )

    view.zoomTo()
    view.render()
    return view


In [17]:
import numpy as np
import networkx as nx
from scipy.spatial import KDTree
from Bio.PDB import PDBParser

def build_geometry_graph_from_pdb(pdb_path, chain="A", cutoff=8.0, atom_name="CA"):
    """
    Build a residue interaction network from a PDB file using CA (or CB fallback) distances.
    Nodes: residue positions (ints)
    Edges: if distance <= cutoff, with attributes distance, weight=1/distance, resistance=distance
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("af", pdb_path)

    # Use first model
    model = next(structure.get_models())

    # Pick chain (AlphaFold is usually 'A')
    if chain not in model:
        chain_obj = next(model.get_chains())
        chain = chain_obj.id  # fallback
    else:
        chain_obj = model[chain]


    coords = []
    meta = []

    for res in chain_obj:
        # standard residues only
        if res.id[0] != " ":
            continue

        resi = int(res.id[1])
        resn = res.resname

        # choose atom
        atom = None
        if atom_name in res:
            atom = res[atom_name]
        elif atom_name == "CB" and "CA" in res:
            atom = res["CA"]
        elif atom_name == "CA":
            # CA required; skip if missing
            continue
        else:
            # fallback to CA if present
            atom = res["CA"] if "CA" in res else None

        if atom is None:
            continue

        coords.append(atom.coord.astype(float))
        meta.append({"Chain": chain, "Residue": resi, "ResName": resn})

    coords = np.asarray(coords, dtype=float)
    if len(coords) == 0:
        raise ValueError("No residue coordinates found. Check chain/atom_name.")

    nodes = [(m["Chain"], int(m["Residue"])) for m in meta]
    tree = KDTree(coords)

    G = nx.Graph(layer=f"geometry:{atom_name}_cut{cutoff}", chain=chain, pdb=pdb_path)

    for n, m in zip(nodes, meta):
        G.add_node(n, **m)

    for i in range(len(nodes)):
        idxs = tree.query_ball_point(coords[i], cutoff)
        for j in idxs:
            if j <= i:
                continue
            d = float(np.linalg.norm(coords[i] - coords[j]))
            w = 1.0 / max(d, 1e-6)
            G.add_edge(nodes[i], nodes[j], weight=w, distance=d, resistance=1.0 / max(w, 1e-9))

    return G, meta


In [18]:
from pyvis.network import Network

def nx_rin_to_pyvis_default(
    G,
    ptm_positions=None,
    mutation_positions=None,
    out_html="rin_pyvis.html",
    height="600px",
    width="100%",
    default_color="#B0B0B0",   # light grey
    ptm_color="#1f77b4",       # blue
    mut_color="#d62728",       # red
    both_color="#9467bd",      # purple
    node_size=30,
    ptm_size=35,
    mut_size=35,
    both_size=40,
    select_menu=True,
    filter_menu=False,
    highlight_resns=None,
    highlight_resi=None
):
    ptm_set = set(int(x) for x in (ptm_positions or []))
    mut_set = set(int(x) for x in (mutation_positions or []))
    hl_resns = set((highlight_resns or ()))
    hl_resi = set(int(x) for x in (highlight_resi or []))

    net = Network(
        height=height,
        width=width,
        directed=False,
        notebook=True,
        cdn_resources="in_line",
        select_menu=select_menu,
        filter_menu=filter_menu
    )

    net.set_options("""
    {
      "groups": {
        "PTM": {
          "color": {
            "background": "#d62728",
            "border": "#d62728",
            "highlight": { "background": "#d62728", "border": "#d62728" },
            "hover":     { "background": "#d62728", "border": "#d62728" }
          }
        },
        "Mutation": {
          "color": {
            "background": "#1f77b4",
            "border": "#1f77b4",
            "highlight": { "background": "#1f77b4", "border": "#1f77b4" },
            "hover":     { "background": "#1f77b4", "border": "#1f77b4" }
          }
        },
        "PTM+Mutation": {
          "color": {
            "background": "#2ca02c",
            "border": "#2ca02c",
            "highlight": { "background": "#2ca02c", "border": "#2ca02c" },
            "hover":     { "background": "#2ca02c", "border": "#2ca02c" }
          }
        },
        "Other": {
          "color": {
            "background": "#9FA8B0",
            "border": "#9FA8B0",
            "highlight": { "background": "#9FA8B0", "border": "#9FA8B0" },
            "hover":     { "background": "#9FA8B0", "border": "#9FA8B0" }
          }
        }
      }
    }
    """)




    # ---- Nodes ----
    for (ch, resi), attrs in G.nodes(data=True):
        resi = int(resi)
        resn = attrs.get("ResName", "")

        is_ptm = resi in ptm_set
        is_mut = resi in mut_set

        if is_ptm and is_mut:
            bg = both_color
            size = both_size
            group = "PTM+Mutation"
        elif is_mut:
            bg = mut_color
            size = mut_size
            group = "Mutation"
        elif is_ptm:
            bg = ptm_color
            size = ptm_size
            group = "PTM"
        else:
            bg = default_color
            size = node_size
            group = "Other"

        node_id = f"{ch}:{resi}"

        _is_other = (group == "Other")
        _bcol = "#000000" if _is_other else "#333333"
        _bw = 1 if _is_other else 3
        _highlighted = (str(resn).upper() in hl_resns) or (int(resi) in hl_resi)
        _nkw = {}
        if _highlighted:
            if _bw == 0:
                _bcol = "#000000"
            _bw = max(_bw, 3)
            _nkw["shapeProperties"] = {"borderDashes": [6, 4]}
        net.add_node(
            node_id,
            label=f"{resn} {resi}" if resn else str(resi),
            title=f"{resn}:{ch}:{resi}:{group}" + (" | selected" if _highlighted else ""),
            color={
                "background": bg,
                "border": _bcol,
                "highlight": {"background": bg, "border": ("#000000" if not _is_other else _bcol)},
                "hover": {"background": bg, "border": ("#000000" if not _is_other else _bcol)}
            },
            size=size,
            group=group,
            font={"size": 12},
            borderWidth=_bw,
            **_nkw
        )

    # ---- Edges ----
    for (ch1, r1), (ch2, r2), eattrs in G.edges(data=True):
        a = f"{ch1}:{int(r1)}"
        b = f"{ch2}:{int(r2)}"
        dist = eattrs.get("distance", None)

        net.add_edge(
            a, b,
            color="#A9A9A9",
            title=f"distance: {dist:.2f} Å" if dist is not None else ""
        )

    html = net.generate_html()
    with open(out_html, "w", encoding="utf-8") as f:
        f.write(html)
    
    return out_html


In [19]:
import os
import subprocess
import tempfile
from Bio.PDB import PDBParser, PDBIO, Select
import nglview as nv
import ipywidgets as widgets
from IPython.display import display, clear_output

def save_upload(upload_widget, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    v = upload_widget.value
    if not v:
        raise ValueError("No file uploaded")

    # Newer ipywidgets: tuple/list of dicts
    if isinstance(v, (tuple, list)):
        fileinfo = v[0]
        name = fileinfo.get("name", "upload.pdb")
        content = fileinfo["content"]

    # Older ipywidgets: dict name -> fileinfo
    elif isinstance(v, dict):
        name, fileinfo = next(iter(v.items()))
        content = fileinfo["content"]

    else:
        raise TypeError(f"Unexpected upload_widget.value type: {type(v)}")

    path = os.path.join(out_dir, name)
    with open(path, "wb") as f:
        f.write(content)
    return path



def chain_range_from_pdb(pdb_path, chain_id):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("X", pdb_path)
    residues = [
        res.id[1]
        for model in structure
        for chain in model
        if chain.id == chain_id
        for res in chain
        if res.id[0] == " "
    ]
    if not residues:
        raise ValueError(f"No residues found for chain {chain_id}")
    return min(residues), max(residues)


class ChainRangeSelect(Select):
    def __init__(self, chain_id, start, end):
        self.chain_id = chain_id
        self.start = start
        self.end = end

    def accept_chain(self, chain):
        return chain.id == self.chain_id

    def accept_residue(self, residue):
        r = residue.id[1]
        return (self.start <= r <= self.end)

def run_tmalign_write(pdb1, pdb2, out_dir, out_name):
    os.makedirs(out_dir, exist_ok=True)
    cmd = ["TM-align", os.path.abspath(pdb1), os.path.abspath(pdb2), "-o", out_name]
    res = subprocess.run(cmd, cwd=out_dir, capture_output=True, text=True, check=True)

    candidates = [
        os.path.join(out_dir, out_name + "_all_atm"),  # full atoms, whole chains (best for cartoon)
        os.path.join(out_dir, out_name + "_atm"),
        os.path.join(out_dir, out_name + "_all"),
        os.path.join(out_dir, out_name),
        os.path.join(out_dir, out_name + ".pdb"),
        os.path.join(out_dir, "TM_sup.pdb"),
    ]
    out_pdb = next((c for c in candidates if os.path.exists(c)), None)
    if out_pdb is None:
        raise RuntimeError(f"No TM-align output found. Files: {os.listdir(out_dir)}")
    return out_pdb, res.stdout

import nglview as nv

def visualize_ngl(pdb_ref, pdb_aligned, selection="protein"):
    view = nv.NGLWidget()

    # CRITICAL: disable default rainbow reps
    view.add_component(pdb_ref, ext="pdb", defaultRepresentation=False)
    view.add_component(pdb_aligned, ext="pdb", defaultRepresentation=False)

    view.clear_representations()

    # Reference (blue, translucent)
    view.add_cartoon(
        component=0,
        selection=selection,
        colorScheme="uniform",
        colorValue="blue",
        opacity=0.7
    )

    # Aligned (red, solid)
    view.add_cartoon(
        component=1,
        selection=selection,
        colorScheme="uniform",
        colorValue="red",
        opacity=1.0
    )

    view.center()
    return view


def run_tmalign_matrix(pdb1, pdb2, out_dir, tag="tm"):
    """Run TM-align and output the rotation matrix that rotates Chain_1 onto Chain_2."""
    os.makedirs(out_dir, exist_ok=True)
    matrix_path = os.path.join(out_dir, f"matrix_{tag}.txt")
    cmd = ["TM-align", os.path.abspath(pdb1), os.path.abspath(pdb2), "-m", os.path.abspath(matrix_path)]
    res = subprocess.run(cmd, cwd=out_dir, capture_output=True, text=True, check=True)
    if not os.path.exists(matrix_path):
        raise RuntimeError(f"TM-align did not write a matrix file. Files: {os.listdir(out_dir)}")
    return matrix_path, res.stdout


def _read_tmalign_matrix(matrix_path):
    """Parse a TM-align -m matrix file.

    Returns (t, u) where t has shape (3,) and u has shape (3, 3).
    Transform applied to Chain_1 coordinates: X_new = t + u . X_old.
    Handles both 0-based (C++) and 1-based (Fortran) row indexing.
    """
    import numpy as np
    rows = []
    with open(matrix_path) as fh:
        for line in fh:
            parts = line.split()
            if len(parts) == 5:
                try:
                    rows.append([float(p) for p in parts])
                except ValueError:
                    continue
    if len(rows) < 3:
        raise RuntimeError(f"Could not parse TM-align matrix rows ({len(rows)} found).")
    rows = rows[:3]
    base = min(int(round(r[0])) for r in rows)  # 0 or 1
    t = np.zeros(3)
    u = np.zeros((3, 3))
    for r in rows:
        idx = int(round(r[0])) - base
        if 0 <= idx <= 2:
            t[idx] = r[1]
            u[idx, 0] = r[2]
            u[idx, 1] = r[3]
            u[idx, 2] = r[4]
    return t, u


def _transform_pdb_with_tmalign_matrix(in_pdb, out_pdb, matrix_path):
    """Apply the TM-align rotation/translation to every atom of in_pdb, write out_pdb."""
    import numpy as np
    t, u = _read_tmalign_matrix(matrix_path)
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("X", in_pdb)
    for atom in structure.get_atoms():
        xyz = np.asarray(atom.coord, dtype=float)
        atom.set_coord(t + u.dot(xyz))
    io = PDBIO()
    io.set_structure(structure)
    io.save(out_pdb)
    return out_pdb


def _ca_coords_by_resnum(pdb_path):
    """Return (resnums, coords) of CA atoms in the first model of a PDB file."""
    import numpy as np
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("X", pdb_path)
    resn, xyz = [], []
    for model in structure:
        for chain in model:
            for res in chain:
                if res.id[0] != " ":
                    continue
                if "CA" in res:
                    resn.append(int(res.id[1]))
                    xyz.append(res["CA"].coord)
        break  # first model only
    return np.array(resn, dtype=int), (np.array(xyz, dtype=float) if xyz else np.zeros((0, 3)))


def aligned_residues_by_proximity(ref_pdb, aligned_pdb, cutoff=5.0):
    """Residues whose CA lies within `cutoff` Angstrom of any CA in the other
    (already superposed) structure. Returns (ref_resnums, aligned_resnums)."""
    import numpy as np
    rn_ref, xyz_ref = _ca_coords_by_resnum(ref_pdb)
    rn_aln, xyz_aln = _ca_coords_by_resnum(aligned_pdb)
    if len(xyz_ref) == 0 or len(xyz_aln) == 0:
        return sorted(rn_ref.tolist()), sorted(rn_aln.tolist())
    diff = xyz_ref[:, None, :] - xyz_aln[None, :, :]
    dist = np.sqrt((diff * diff).sum(axis=2))
    ref_hits = rn_ref[dist.min(axis=1) <= cutoff]
    aln_hits = rn_aln[dist.min(axis=0) <= cutoff]
    return sorted(int(x) for x in ref_hits), sorted(int(x) for x in aln_hits)


In [20]:
upload1 = widgets.FileUpload(accept=".pdb", multiple=False, description="Upload PDB 1")
upload2 = widgets.FileUpload(accept=".pdb", multiple=False, description="Upload PDB 2")

chain1 = widgets.Text(value="A", description="Chain 1")
chain2 = widgets.Text(value="A", description="Chain 2")

start1 = widgets.IntText(description="Start 1")
end1   = widgets.IntText(description="End 1")
start2 = widgets.IntText(description="Start 2")
end2   = widgets.IntText(description="End 2")

btn_range = widgets.Button(description="Auto-fill ranges")
btn_run = widgets.Button(description="Align + Visualize", button_style="primary")

out = widgets.Output()
workdir = tempfile.mkdtemp(prefix="tmalign_upload_tool_")


In [21]:
# Original TM-align upload-only autofill callback replaced by enhanced dropdown-aware callback in the main app cell.

In [22]:
# Original TM-align upload-only run callback replaced by enhanced dropdown-aware callback in the main app cell.

In [23]:
tmalign_ui = widgets.VBox([
    widgets.HBox([upload1, upload2]),
    widgets.HBox([chain1, start1, end1]),
    widgets.HBox([chain2, start2, end2]),
    widgets.HBox([btn_range, btn_run]),
    out,
])


In [24]:
import os, tempfile, traceback, re, uuid, shutil, html, base64, mimetypes, json
import ipywidgets as w
from IPython.display import display, clear_output, IFrame, HTML, FileLink
import requests
import py3Dmol

# ----------------------------
# Scrollable df helper
# ----------------------------
def display_scrollable_df(df, max_height="420px", max_width="100%"):
    if df is None:
        return
    html_table = df.to_html(index=False, escape=False)
    css = f"""
    <style>
      .scroll-df-wrap {{
        width: 100%;
        max-width: {max_width};
        max-height: {max_height};
        overflow-x: auto;
        overflow-y: auto;
        border: 1px solid #e0e0e0;
        border-radius: 6px;
        box-sizing: border-box;
      }}
      .scroll-df-wrap table {{
        min-width: 100%;
        border-collapse: collapse;
        font-size: 13px;
      }}
      .scroll-df-wrap th,
      .scroll-df-wrap td {{
        padding: 6px 8px;
        border-bottom: 1px solid #eee;
        text-align: left;
        white-space: nowrap;
      }}
      .scroll-df-wrap thead th {{
        position: sticky;
        top: 0;
        background: #fafafa;
        z-index: 2;
        border-bottom: 1px solid #ddd;
      }}
    </style>
    """
    display(HTML(css + f"<div class='scroll-df-wrap'>{html_table}</div>"))

def _err(out: w.Output, e: Exception):
    with out:
        print("❌ Error:", e)
        traceback.print_exc()

def _need_acc(out: w.Output):
    if not state.get("acc"):
        with out:
            print("Set a UniProt accession first (top bar).")
        return True
    return False


# ----------------------------
# Export helpers
# ----------------------------
def _safe_token(x):
    x = str(x or "export")
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", x).strip("_") or "export"

def _view_to_html(view, title="Scop3P structure view"):
    """Best-effort HTML export for the currently rendered structure widget."""
    body = ""
    try:
        if hasattr(view, "_make_html"):
            body = view._make_html()
    except Exception:
        body = ""
    if not body:
        try:
            body = view._repr_html_()
        except Exception:
            body = ""
    if not body:
        body = "<p>Could not serialize this interactive widget directly. Re-open the notebook/Voila app to reproduce the view.</p>"
    return f"""<!doctype html>
<html><head><meta charset="utf-8"><title>{html.escape(title)}</title></head>
<body><h3>{html.escape(title)}</h3>{body}</body></html>"""


def _display_direct_download(path, label=None):
    """Show a real one-click download link using a data URI.

    This works better in Voila than a plain FileLink because the browser gets a
    download= filename and starts downloading immediately when clicked.
    """
    if not path or not os.path.exists(path):
        return
    label = label or os.path.basename(path)
    fname = os.path.basename(path)
    mime = mimetypes.guess_type(fname)[0] or "application/octet-stream"
    try:
        with open(path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("ascii")
        href = f"data:{mime};base64,{b64}"
        display(HTML(
            f"<div style='margin:4px 0;'>"
            f"<a download='{html.escape(fname)}' href='{href}' "
            f"style='display:inline-block;padding:6px 10px;border:1px solid #aaa;"
            f"border-radius:4px;text-decoration:none;background:#f8f8f8;'>"
            f"⬇ Download {html.escape(label)}</a>"
            f"</div>"
        ))
    except Exception:
        # Fallback for very large files or unusual environments.
        display(FileLink(path, result_html_prefix=f"{label}: "))




def _html_iframe_from_file(path, width="100%", height=650):
    """Embed a local HTML file as srcdoc so Voila/Jupyter does not request /files/... URLs."""
    if not path or not os.path.exists(path):
        return HTML("<b>HTML view missing.</b>")
    txt = open(path, "r", encoding="utf-8", errors="ignore").read()
    # srcdoc needs escaping, but keep scripts runnable.
    return HTML(
        f"<iframe style='width:{width};height:{int(height)}px;border:1px solid #ddd;border-radius:6px;' "
        f"srcdoc=\"{html.escape(txt, quote=True)}\"></iframe>"
    )
def _ngl_selection_from_residues(residues, chain=None):
    """Build a compact NGL selection string from PDB residue numbers.

    NGL uses bare residue numbers/ranges (e.g. "100" or "100-120"), NOT the
    PyMOL/VMD "resi" keyword. Chain is appended as ":A".
    """
    vals = sorted({int(x) for x in residues if x is not None})
    if not vals:
        return ""
    parts = []
    start = prev = vals[0]
    for x in vals[1:]:
        if x == prev + 1:
            prev = x
        else:
            parts.append((start, prev))
            start = prev = x
    parts.append((start, prev))
    ranges = [f"{a}" if a == b else f"{a}-{b}" for a, b in parts]
    sel = " or ".join(ranges)
    ch = (chain or "").strip().upper()[:1]
    if ch:
        sel = f"({sel}) and :{ch}"
    return sel
def _mapped_site_residues_for_export(mode="both", pdb_to_uniprot=None):
    """Return PDB residue IDs for selected PTM/variant sites using the same logic as the viewer overlay."""
    ptms = _collect_ptms_from_table(state.get("ptm_table"))
    muts = set(_collect_variant_positions(state.get("variants_df")))
    ptm_set = {int(pos) for pos, _res in ptms if pos is not None}
    mut_set = {int(x) for x in muts if x is not None}
    m = (mode or "both").lower()
    if m == "ptm":
        uni_positions = sorted(ptm_set)
    elif m in {"mut", "mutation", "variant"}:
        uni_positions = sorted(mut_set)
    elif m in {"overlap", "ptm+mutation", "ptm_variant"}:
        uni_positions = sorted(ptm_set & mut_set)
    elif m in {"none", ""}:
        uni_positions = []
    else:
        uni_positions = sorted(ptm_set | mut_set)

    if pdb_to_uniprot:
        uniprot_to_pdb = {}
        for pdb_resi, uni_pos in pdb_to_uniprot.items():
            try:
                uniprot_to_pdb.setdefault(int(uni_pos), []).append(int(pdb_resi))
            except Exception:
                continue
        mapped = []
        for up in uni_positions:
            mapped.extend(uniprot_to_pdb.get(int(up), []))
        return sorted(set(mapped))
    return sorted(set(uni_positions))

def _site_representation_name(style):
    s = (style or "stick").lower()
    if s in {"sphere", "spheres"}:
        return "spacefill"
    if s in {"ballstick", "ball+stick", "ball-and-stick"}:
        return "ball+stick"
    return "licorice"


def _normalize_ngl_rep_params(params):
    """Normalize representation color params for standalone NGL.

    nglview accepts color/color_value/color_scheme aliases, but plain NGL.js
    expects colorScheme/colorValue for fixed colors.  Without this, exported
    or iframe views can silently fall back to default structure coloring.
    """
    params = dict(params or {})
    if "color_scheme" in params and "colorScheme" not in params:
        params["colorScheme"] = params.pop("color_scheme")
    if "color_value" in params and "colorValue" not in params:
        params["colorValue"] = params.pop("color_value")

    # Fixed named/hex colors must be passed as uniform colorValue in NGL.js.
    fixed = params.pop("color", None)
    if fixed is not None and "colorScheme" not in params:
        params["colorScheme"] = "uniform"
        params["colorValue"] = fixed

    # If a uniform colorScheme has no colorValue but a fixed color was supplied, preserve it.
    if fixed is not None and params.get("colorScheme") == "uniform" and "colorValue" not in params:
        params["colorValue"] = fixed
    return params

def _normalize_ngl_components(components):
    comps = []
    for comp in components or []:
        comp = dict(comp or {})
        reps = []
        for rep in comp.get("representations", []) or []:
            rep = dict(rep or {})
            rep["params"] = _normalize_ngl_rep_params(rep.get("params", {}))
            reps.append(rep)
        comp["representations"] = reps
        comps.append(comp)
    return comps

def _write_standalone_ngl_html(
    html_path,
    pdb_paths,
    title="Scop3P exported structure",
    components=None,
    note="",
):
    """Write a true standalone NGL HTML file, embedding structure text and representation instructions.

    This does not attempt to serialize ipywidgets/nglview. Instead it recreates the current
    visual scene in plain JavaScript NGL, like the peptide mapper notebook.
    """
    if isinstance(pdb_paths, (str, os.PathLike)):
        pdb_paths = [str(pdb_paths)]
    pdb_payload = []
    for p in pdb_paths:
        if not p or not os.path.exists(p):
            continue
        pdb_payload.append({
            "name": os.path.basename(p),
            "text": open(p, "r", encoding="utf-8", errors="ignore").read(),
            "ext": "pdb" if str(p).lower().endswith(".pdb") else "cif",
        })
    if not pdb_payload:
        raise RuntimeError("No structure file available for HTML export.")

    components = components or []
    while len(components) < len(pdb_payload):
        components.append({"representations": [{"type": "cartoon", "params": {"colorScheme": "uniform", "colorValue": "grey"}}]})
    components = _normalize_ngl_components(components)

    payload = {"structures": pdb_payload, "components": components, "title": title, "note": note}
    html_txt = f"""<!doctype html>
<html>
<head>
  <meta charset="utf-8"/>
  <title>{html.escape(title)}</title>
  <style>
    body {{ margin: 0; font-family: Arial, sans-serif; }}
    #viewport {{ width: 100vw; height: 100vh; }}
    #panel {{
      position: absolute; top: 10px; left: 10px; z-index: 10;
      background: rgba(255,255,255,0.92); padding: 10px 12px; border-radius: 8px;
      max-width: 620px; box-shadow: 0 1px 6px rgba(0,0,0,0.18);
      font-size: 13px;
    }}
  </style>
  <script src="https://unpkg.com/ngl@latest/dist/ngl.js"></script>
</head>
<body>
  <div id="panel">
    <b>{html.escape(title)}</b><br/>
    <span>{html.escape(note or "Standalone exported NGL view")}</span>
  </div>
  <div id="viewport"></div>
  <script>
    const payload = {json.dumps(payload)};
    const stage = new NGL.Stage("viewport", {{ backgroundColor: "white" }});
    window.addEventListener("resize", () => stage.handleResize(), false);

    async function loadComponent(item, compConfig) {{
      const blob = new Blob([item.text], {{type: "text/plain"}});
      const comp = await stage.loadFile(blob, {{ ext: item.ext || "pdb", name: item.name }});
      const reps = (compConfig && compConfig.representations) || [];
      for (const rep of reps) {{
        comp.addRepresentation(rep.type, rep.params || {{}});
      }}
      return comp;
    }}

    async function loadAllSequentially() {{
      for (let i = 0; i < payload.structures.length; i++) {{
        await loadComponent(payload.structures[i], payload.components[i] || {{}});
      }}
      stage.autoView();
    }}
    loadAllSequentially();
  </script>
</body>
</html>
"""
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html_txt)
    return html_path

def _export_standalone_html(prefix, view_key, pdb_key=None, pdbid_key=None):
    """Create standalone HTML for the known Scop3P structure tabs."""
    prefix = _safe_token(prefix)
    html_path = os.path.join(state["workdir"], f"{prefix}.html")
    view_key = str(view_key or "")

    # Tab 3: structure + PTM/variant overlays
    if view_key == "tab3_last_view":
        pdb_path = state.get("tab3_last_pdb_path")
        chain = state.get("tab3_last_chain") or "A"
        pdb_to_uniprot = None
        if state.get("tab3_last_pdb_id"):
            try:
                pdb_to_uniprot, _ = _build_structure_position_maps(
                    state.get("tab3_last_pdb_id"), chain=chain, uni_range=None, pdb_path=pdb_path
                )
            except Exception:
                pdb_to_uniprot = None

        # Build semantic site groups so Tab 3 export preserves the same colours as the live viewer.
        # PTM: residue-specific colour; mutation: red; PTM+mutation: purple.
        ptms = _collect_ptms_from_table(state.get("ptm_table"))
        muts = set(_collect_variant_positions(state.get("variants_df")))
        ptm_pos_to_res = {int(pos): str(res).upper() for pos, res in ptms if pos is not None}
        ptm_set = set(ptm_pos_to_res.keys())
        mut_set = {int(x) for x in muts if x is not None}
        mode = (map_mode.value or "both").lower()
        if mode == "ptm":
            uni_positions = sorted(ptm_set)
        elif mode in {"mut", "mutation", "variant"}:
            uni_positions = sorted(mut_set)
        else:
            uni_positions = sorted(ptm_set | mut_set)

        # UniProt -> displayed residue numbering, if mapping exists.
        uniprot_to_pdb = {}
        if pdb_to_uniprot:
            for pdb_resi, uni_pos in pdb_to_uniprot.items():
                try:
                    uniprot_to_pdb.setdefault(int(uni_pos), []).append(int(pdb_resi))
                except Exception:
                    pass

        color_to_residues = {}
        for up in uni_positions:
            in_ptm = up in ptm_set
            in_mut = up in mut_set
            if mode == "ptm" and not in_ptm:
                continue
            if mode in {"mut", "mutation", "variant"} and not in_mut:
                continue
            if in_ptm and in_mut:
                color = "purple"
            elif in_mut:
                color = "red"
            else:
                color = PTM_COLOR.get(ptm_pos_to_res.get(up, ""), "orange")
            mapped = uniprot_to_pdb.get(up, [up]) if uniprot_to_pdb else [up]
            color_to_residues.setdefault(color, set()).update(mapped)

        reps = [{"type": "cartoon", "params": {"sele": f":{chain}" if chain else "protein", "color": "lightgrey"}}]
        rep_type = _site_representation_name(site_rep_mode.value)
        for color, residues in color_to_residues.items():
            site_sel = _ngl_selection_from_residues(sorted(residues), chain=chain)
            if site_sel:
                reps.append({"type": rep_type, "params": {"sele": site_sel, "color": color}})
        return _write_standalone_ngl_html(
            html_path, pdb_path, title=prefix,
            components=[{"representations": reps}],
            note=f"Tab 3 export | sites: {map_mode.label if hasattr(map_mode,'label') else map_mode.value} | style: {site_rep_mode.value}"
        )

    # Tab 4: Bio2Byte colored structure + marker overlays
    if view_key == "tab4_last_view":
        pdb_path = state.get("tab4_last_colored_pdb_path") or state.get("tab4_last_source_pdb_path")
        chain = None
        pdb_to_uniprot = None
        try:
            if b2b_source.value == "af":
                chain = "A"
            else:
                # Current Tab 4 widget names are b2b_chain_text / b2b_pdb_text / b2b_pdb_dropdown.
                # Older export code still referenced removed names, which caused the standalone export to fail.
                chain = (globals().get("b2b_chain_text").value or "").strip().upper()[:1] or None
                pdbid = ((globals().get("b2b_pdb_dropdown").value or globals().get("b2b_pdb_text").value or "").strip().upper())
                if pdbid:
                    pdb_to_uniprot, _ = _build_structure_position_maps(pdbid, chain=chain, uni_range=None, pdb_path=pdb_path)
        except Exception:
            pdb_to_uniprot = None
        sel = f":{chain}" if chain else "protein"
        # Match the interactive Bio2Byte viewer: the exported PDB already contains 0-100 scaled values in B-factors.
        # Fix the color domain so standalone NGL does not rescale differently on export.
        reps = [{"type": "cartoon", "params": {"sele": sel, "colorScheme": "bfactor", "colorDomain": [0, 100]}}]
        if (b2b_site_overlay.value or "none") != "none":
            sites = _mapped_site_residues_for_export(b2b_site_overlay.value, pdb_to_uniprot=pdb_to_uniprot)
            site_sel = _ngl_selection_from_residues(sites, chain=chain)
            if site_sel:
                # Marker only: representation marks selected sites; color remains controlled by Bio2Byte/B-factor.
                reps.append({"type": _site_representation_name(b2b_site_style.value), "params": {"sele": site_sel, "color": "#7a7a7a"}})
        return _write_standalone_ngl_html(
            html_path, pdb_path, title=prefix,
            components=[{"representations": reps}],
            note=f"Tab 4 Bio2Byte export | property: {b2b_metric.value} | sites: {b2b_site_overlay.value}"
        )

    # Tab 6: TM-align alignment (reference + transformed structure 1, two components)
    if view_key == "tab6_last_view":
        ref = state.get("tab6_last_ref_pdb_path")
        aln = state.get("tab6_last_aligned_pdb_path")
        if (tm_view_region.value or "all").lower() == "aligned":
            sel_ref_base = _aligned_region_selection()
            sel_aln_base = _aligned_region_selection()
        else:
            sel_ref_base = "protein"
            sel_aln_base = "protein"
        reps_ref = [{"type": "cartoon", "params": {"sele": sel_ref_base, "color": "#1f77b4", "opacity": 0.6}}]
        reps_aln = [{"type": "cartoon", "params": {"sele": sel_aln_base, "color": "#d62728", "opacity": 0.85}}]
        if (tm_site_overlay.value or "none") != "none":
            sites = _mapped_site_residues_for_export(tm_site_overlay.value, pdb_to_uniprot=None)
            if (tm_view_region.value or "all").lower() == "aligned":
                rng = state.get("tab6_aligned_range")
                if rng:
                    lo, hi = rng
                    sites = [s for s in sites if lo <= int(s) <= hi]
            rep = _site_representation_name(tm_site_style.value)
            site_sel = _ngl_selection_from_residues(sites, chain=None)
            if site_sel:
                target = (tm_site_target.value or "both").lower()
                if target in ("both", "s1"):
                    reps_aln.append({"type": rep, "params": {"sele": site_sel, "color": "#7a7a7a"}})
                if target in ("both", "s2"):
                    reps_ref.append({"type": rep, "params": {"sele": site_sel, "color": "#7a7a7a"}})
        return _write_standalone_ngl_html(
            html_path, [ref, aln], title=prefix,
            components=[{"representations": reps_ref}, {"representations": reps_aln}],
            note="Tab 6 TM-align export | blue = reference/structure 2, red = aligned/structure 1, grey = selected sites"
        )

    # Generic fallback for any other structure view
    pdb_path = state.get(pdb_key) if pdb_key else None
    if pdb_path:
        return _write_standalone_ngl_html(html_path, pdb_path, title=prefix)
    raise RuntimeError("No standalone export recipe available for this view.")


def _export_current_view(prefix, view_key, pdb_key=None, pdbid_key=None, out=None, also_cif=True):
    """Export last rendered view as HTML and current/mapped structure as PDB/CIF where available."""
    if out is None:
        out = top_out
    os.makedirs(state.get("workdir", tempfile.gettempdir()), exist_ok=True)
    prefix = _safe_token(prefix)
    exported = []

    view = state.get(view_key)
    if view is not None:
        try:
            html_path = _export_standalone_html(prefix, view_key, pdb_key=pdb_key, pdbid_key=pdbid_key)
            exported.append(("HTML view", html_path))
        except Exception as e:
            # Last-resort fallback only; normal structure tabs use true standalone NGL HTML.
            html_path = os.path.join(state["workdir"], f"{prefix}.html")
            try:
                with open(html_path, "w", encoding="utf-8") as f:
                    f.write(_view_to_html(view, title=prefix))
                exported.append(("HTML view", html_path))
                with out:
                    print("⚠️ Standalone NGL export failed, used widget HTML fallback:", e)
            except Exception as e2:
                with out:
                    print("⚠️ HTML export failed:", e2)

    pdb_path = state.get(pdb_key) if pdb_key else None
    if pdb_path and os.path.exists(pdb_path):
        dst = os.path.join(state["workdir"], f"{prefix}.pdb")
        try:
            if os.path.abspath(pdb_path) != os.path.abspath(dst):
                shutil.copyfile(pdb_path, dst)
            exported.append(("PDB structure", dst))
        except Exception as e:
            with out:
                print("⚠️ PDB export failed:", e)

    pdb_id = state.get(pdbid_key) if pdbid_key else None
    if also_cif and pdb_id:
        try:
            cif = _download_pdbe_updated_cif(str(pdb_id), state["workdir"])
            if cif and os.path.exists(cif):
                dst = os.path.join(state["workdir"], f"{prefix}.cif")
                if os.path.abspath(cif) != os.path.abspath(dst):
                    shutil.copyfile(cif, dst)
                exported.append(("mmCIF structure", dst))
        except Exception as e:
            with out:
                print("⚠️ CIF export failed:", e)

    with out:
        if not exported:
            print("Nothing to export yet. Render/build the view first.")
        else:
            print("✅ Exported. Click a button below to download immediately:")
            for label, path in exported:
                _display_direct_download(path, label)

def download_alphafold_pdb(accession: str, out_dir: str) -> str:
    """Download AlphaFold PDB (model_v6) to out_dir and return file path."""
    os.makedirs(out_dir, exist_ok=True)
    url = f"https://alphafold.ebi.ac.uk/files/AF-{accession}-F1-model_v6.pdb"
    fp = os.path.join(out_dir, f"AF-{accession}-F1-model_v6.pdb")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    with open(fp, "wb") as f:
        f.write(r.content)
    return fp

# ----------------------------
# Top bar: select protein
# ----------------------------
acc_input = w.Text(description="UniProt:", placeholder="e.g., P07949", layout=w.Layout(width="320px"))
btn_set = w.Button(description="Set protein", button_style="info")
lbl = w.HTML("<b>Current:</b> (not set)")
top_out = w.Output(layout={"border":"1px solid #ddd","padding":"6px"})

# ----------------------------
# UniProt PDB cross-references (for Tab 3/4 dropdowns)
# ----------------------------
def fetch_uniprot_pdb_xrefs(accession: str):
    """Fetch UniProtKB JSON and parse PDB IDs + chain->(start,end) residue ranges when available."""
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    data = r.json()

    refs = []
    for x in data.get("uniProtKBCrossReferences", []) or []:
        if x.get("database") != "PDB":
            continue
        pdb_id = x.get("id")
        props = {p.get("key"): p.get("value") for p in (x.get("properties") or []) if isinstance(p, dict)}
        chains_raw = props.get("Chains") or props.get("Chain") or ""
        chain_ranges = {}

        # Chains formats seen in UniProt:
        #  "A=1-200" or "A=1-200, B=5-150" or "A=10-50; 70-120" or "A/B=1-120"
        if chains_raw:
            parts = [p.strip() for p in re.split(r",\s*(?=[A-Za-z0-9/]+=)", chains_raw) if p.strip()]
            for part in parts:
                if "=" not in part:
                    continue
                lhs, rngs = part.split("=", 1)
                lhs = lhs.strip()
                rngs = rngs.strip()

                starts, ends = [], []
                for m in re.finditer(r"(\d+)\s*-\s*(\d+)", rngs):
                    starts.append(int(m.group(1)))
                    ends.append(int(m.group(2)))

                # chain part may be A or A/B
                chains = [c.strip().upper()[:1] for c in re.split(r"[\/\s]+", lhs) if c.strip()]
                if starts and ends:
                    for ch in chains:
                        chain_ranges[ch] = (min(starts), max(ends))
                else:
                    for ch in chains:
                        chain_ranges.setdefault(ch, None)

        refs.append({
            "pdb_id": pdb_id,
            "chain_ranges": chain_ranges,   # may include None for range n/a
            "raw_chains": chains_raw,
            "method": props.get("Method"),
            "resolution": props.get("Resolution"),
        })
    return refs

def on_set(_):
    with top_out:
        clear_output()
        acc = acc_input.value.strip()
        if not acc:
            print("Please enter a UniProt accession.")
            return

        state["acc"] = acc
        lbl.value = f"<b>Current:</b> {acc}"

        # Fetch compact UniProt protein metadata for the Tab 1 information card
        try:
            state["protein_info"] = fetch_uniprot_protein_info(acc)
            protein_info_box.value = render_protein_info_html(state["protein_info"], state.get("ptm_source_label"))
        except Exception as e:
            state["protein_info"] = None
            protein_info_box.value = f"<div style='color:#a66;'>Protein information fetch failed: {_safe_html(e)}</div>"

        # Fetch UniProt PDB cross-references for dropdowns (Tabs 3/4/5/6)
        try:
            state["uniprot_pdb_refs"] = fetch_uniprot_pdb_xrefs(acc)
        except Exception as e:
            state["uniprot_pdb_refs"] = []
            print(f"⚠️ UniProt PDB cross-ref fetch failed: {e}")

        print(f"✅ Protein set: {acc}")
        print(f"Session workdir: {state['workdir']}")

        # Refresh Tab 3 dropdowns if present
        try:
            _refresh_pdb_dropdowns()
        except Exception:
            pass

        # Refresh Tab 4 dropdowns if present
        try:
            _refresh_b2b_pdb_dropdowns()
        except Exception:
            pass

        # Refresh Tab 5 dropdowns if present
        try:
            _refresh_rin_pdb_dropdowns()
        except Exception:
            pass

        # Refresh Tab 6 dropdowns if present
        try:
            _refresh_tm_pdb_dropdowns()
        except Exception:
            pass

        # Data-availability probe: PDB count is known now; AlphaFold via a HEAD request.
        state.setdefault("avail", {})
        state["avail"]["pdb"] = len(state.get("uniprot_pdb_refs") or [])
        try:
            state["avail"]["af"] = _alphafold_available(acc)
        except Exception:
            state["avail"]["af"] = None
        state["avail"]["ptm"] = None   # reset for new protein (fetched in Tab 1)
        state["avail"]["var"] = None   # reset for new protein (fetched in Tab 2)
        render_availability_strip()
        _apply_structure_gating()
        if state["avail"].get("af") is False and state["avail"]["pdb"] == 0:
            print("\u26a0\ufe0f No AlphaFold model or UniProt PDB structures found for this protein.")
            print("   Auto-fetch controls are disabled; use 'Upload PDB' (Tabs 5/6) or type a PDB ID (Tabs 3/4).")

btn_set.on_click(on_set)
top = w.VBox([w.HBox([acc_input, btn_set, lbl]), top_out])

# ----------------------------
# TAB 1: PTMs (Scop3P + UniProt)
# ----------------------------
out_ptm = w.Output(layout={"border":"1px solid #ddd","padding":"6px"})
btn_fetch_ptm = w.Button(description="Fetch PTMs", button_style="warning")
btn_show_ptm = w.Button(description="Show table")
include_uniprot_ptm = w.Checkbox(
    value=False,
    description="Also include all UniProt PTMs",
    indent=False,
    layout=w.Layout(width="420px")
)
ptm_header = w.HTML("<h3>Scop3P PTMs</h3>")
protein_info_box = w.HTML(render_protein_info_html(None))
availability_box = w.HTML("")  # persistent data-availability strip (above tabs)

def fetch_ptm(_):
    try:
        with out_ptm:
            clear_output()
            if _need_acc(out_ptm):
                return
            print("Fetching PTMs...")

        acc = state["acc"]

        # Always try Scop3P first. This preserves the original human Scop3P behavior.
        scop3p_res = fetch_protein_modifications(acc)
        scop3p_mods = (scop3p_res or {}).get("modifications", [])
        scop3p_tbl = get_modification_table(scop3p_mods, accession=acc, source="Scop3P")

        # UniProt PTMs are optional when Scop3P data exists, and automatic fallback otherwise.
        use_uniprot = include_uniprot_ptm.value or scop3p_tbl.empty
        if use_uniprot:
            uniprot_tbl = fetch_uniprot_ptms(acc)
        else:
            uniprot_tbl = pd.DataFrame(columns=scop3p_tbl.columns)

        if not scop3p_tbl.empty and not uniprot_tbl.empty:
            ptm_label = "Scop3P + UniProt PTMs"
        elif not scop3p_tbl.empty:
            ptm_label = "Scop3P PTMs"
        elif not uniprot_tbl.empty:
            ptm_label = "UniProt PTMs"
        else:
            ptm_label = "No PTMs found"

        # Merge Scop3P + UniProt by biological site.
        # Scop3P wins for duplicate phosphosites, so names such as "phosphorylation",
        # source keywords such as "UP", and the Scop3P feature label are retained.
        tbl = merge_scop3p_uniprot_ptms(scop3p_tbl, uniprot_tbl)

        state["ptm_json"] = {
            "Scop3P": scop3p_res,
            "UniProt": {"enabled": bool(use_uniprot), "n_features": len(uniprot_tbl)},
        }
        state["ptm_table"] = tbl
        state["ptm_source_label"] = ptm_label
        globals()["modification_table"] = tbl  # legacy compatibility
        state.setdefault("avail", {})["ptm"] = int(len(tbl))
        try:
            render_availability_strip()
        except Exception:
            pass

        ptm_header.value = f"<h3>{_safe_html(ptm_label)}</h3>"
        protein_info_box.value = render_protein_info_html(state.get("protein_info"), ptm_label)

        with out_ptm:
            print(f"✅ Done. Scop3P PTMs: {len(scop3p_tbl)} | UniProt PTMs shown: {len(uniprot_tbl)} | Total shown: {len(tbl)}")
            if not include_uniprot_ptm.value and not scop3p_tbl.empty:
                print("Tip: enable 'Also include all UniProt PTMs' to append all single-site UniProt PTM annotations.")
            display_scrollable_df(tbl, max_height="420px", max_width="95vw")

    except Exception as e:
        _err(out_ptm, e)

def show_ptm(_):
    with out_ptm:
        clear_output()
        if state.get("ptm_table") is None:
            print("No PTM table yet. Click 'Fetch PTMs'.")
            return
        display_scrollable_df(state["ptm_table"], max_height="420px", max_width="95vw")

btn_fetch_ptm.on_click(fetch_ptm)
btn_show_ptm.on_click(show_ptm)
tab1 = w.VBox([
    ptm_header,
    w.HTML("<p>Fetches Scop3P PTMs by default. Enable the UniProt option to append all single-site UniProt PTM annotations, or use UniProt automatically when Scop3P has no PTMs.</p>"),
    include_uniprot_ptm,
    w.HBox([btn_fetch_ptm, btn_show_ptm]),
    out_ptm
])

# ----------------------------
# TAB 2: Variants (UniProt/EBI API)
# ----------------------------
out_var = w.Output(layout={"border":"1px solid #ddd","padding":"6px"})
btn_fetch_var = w.Button(description="Fetch disease-associated variants", button_style="warning")

def fetch_var(_):
    try:
        with out_var:
            clear_output()
            if _need_acc(out_var):
                return
            print("Fetching variants...")

        df = fetch_uniprot_variants_disease(state["acc"])
        state["variants_df"] = df
        n_var = 0 if df is None else int(len(df))
        state.setdefault("avail", {})["var"] = n_var
        try:
            render_availability_strip()
        except Exception:
            pass

        with out_var:
            if n_var == 0:
                print("\u2705 Done. Disease-associated variants: 0")
                print("\u26a0\ufe0f No disease-associated variants found for this protein.")
            else:
                print(f"\u2705 Done. Disease-associated variants: {n_var}")
                display_scrollable_df(df, max_height="420px", max_width="95vw")

    except Exception as e:
        _err(out_var, e)

btn_fetch_var.on_click(fetch_var)
tab2 = w.VBox([w.HTML("<h3>Disease-associated variants</h3>"), btn_fetch_var, out_var])

# ----------------------------
# TAB 3: 3D structure viewer (PDB + AlphaFold)  [NGLVIEW ONLY]
# ----------------------------
import nglview as nv

out_3d = w.Output(layout={"border":"1px solid #ddd","padding":"8px"})

structure_source = w.ToggleButtons(
    options=[("PDB", "pdb"), ("AlphaFold", "af")],
    value="pdb",
    description="Source:"
)

pdb_input   = w.Text(description="PDB:", placeholder="e.g., 2IVT", layout=w.Layout(width="240px"))
chain_input = w.Text(description="Chain:", placeholder="A (optional)", layout=w.Layout(width="200px"))

map_mode = w.Dropdown(
    description="Map:",
    options=[("PTMs", "ptm"), ("Mutations", "mut"), ("Both", "both")],
    value="both",
    layout=w.Layout(width="220px")
)

site_rep_mode = w.Dropdown(
    description="Site overlay:",
    options=[("Sticks/licorice", "stick"), ("Sphere", "sphere"), ("Ball+stick", "ballstick")],
    value="stick",
    layout=w.Layout(width="280px")
)

tab3_viewer_mode = w.ToggleButtons(
    description="Viewer:",
    options=[("Interactive NGL", "ngl"), ("py3Dmol color", "py3dmol")],
    value="ngl",
    layout=w.Layout(width="430px")
)

# Dropdowns populated from UniProt PDB cross-references
pdb_dd   = w.Dropdown(options=[("", "")], value="", description="UniProt PDB:", layout=w.Layout(width="420px"))
chain_dd = w.Dropdown(options=[("", "")], value="", description="Chain/range:", layout=w.Layout(width="420px"))
range_lbl = w.HTML("")

def _refresh_pdb_dropdowns():
    refs = state.get("uniprot_pdb_refs") or []
    pdb_ids = sorted({(r.get("pdb_id") or "").upper() for r in refs if r.get("pdb_id")})
    if not pdb_ids:
        pdb_dd.options = [("", "")]
        pdb_dd.value = ""
        chain_dd.options = [("", "")]
        chain_dd.value = ""
        range_lbl.value = "<i>No UniProt PDB cross-references found.</i>"
        return

    pdb_dd.options = [("", "")] + [(pid, pid) for pid in pdb_ids]
    if pdb_dd.value not in [v for _, v in pdb_dd.options]:
        pdb_dd.value = ""
    _update_chain_dropdown_from_pdb()

def _update_chain_dropdown_from_pdb(*_):
    pid = (pdb_dd.value or "").upper().strip()
    if not pid:
        chain_dd.options = [("", "")]
        chain_dd.value = ""
        range_lbl.value = ""
        return

    refs = state.get("uniprot_pdb_refs") or []
    cr = {}
    for r in refs:
        if (r.get("pdb_id") or "").upper() != pid:
            continue
        for ch, rng in (r.get("chain_ranges") or {}).items():
            cr[ch] = rng

    if not cr:
        chain_dd.options = [("", "")]
        chain_dd.value = ""
        range_lbl.value = "<i>No chain information available in UniProt for this PDB.</i>"
        return

    opts = [("", "")]
    for ch in sorted(cr.keys()):
        rng = cr[ch]
        if rng and len(rng) == 2:
            label = f"{ch} ({rng[0]}-{rng[1]})"
        else:
            label = f"{ch} (range n/a)"
        opts.append((label, ch))
    chain_dd.options = opts
    if chain_dd.value not in [v for _, v in opts]:
        chain_dd.value = ""

    _sync_textboxes_from_dropdowns()

def _sync_textboxes_from_dropdowns(*_):
    if pdb_dd.value:
        pdb_input.value = pdb_dd.value
    if chain_dd.value:
        chain_input.value = chain_dd.value

    # update range label if known
    pid = (pdb_dd.value or "").upper().strip()
    ch = (chain_dd.value or "").strip().upper()[:1] if chain_dd.value else ""
    if pid and ch:
        refs = state.get("uniprot_pdb_refs") or []
        for r in refs:
            if (r.get("pdb_id") or "").upper() != pid:
                continue
            rng = (r.get("chain_ranges") or {}).get(ch)
            if rng and len(rng) == 2:
                range_lbl.value = f"<b>UniProt range:</b> {ch} = {rng[0]}–{rng[1]}"
                return
        range_lbl.value = f"<b>UniProt range:</b> {ch} = n/a"
    else:
        range_lbl.value = ""

pdb_dd.observe(_update_chain_dropdown_from_pdb, names="value")
chain_dd.observe(_sync_textboxes_from_dropdowns, names="value")

btn_fetch_af = w.Button(description="Fetch AlphaFold", button_style="warning")
btn_show_3d  = w.Button(description="Show 3D", button_style="success")
btn_export_tab3 = w.Button(description="Export view/structure", button_style="", tooltip="Export the last Tab 3 view as HTML and structure as PDB/CIF when available")

PTM_COLOR = {"SER": "red", "THR": "green", "TYR": "orange"}

def _download_pdb_to_workdir(pdb_id: str, outdir: str) -> str:
    pdb_id = pdb_id.strip().lower()
    os.makedirs(outdir, exist_ok=True)
    outpath = os.path.join(outdir, f"{pdb_id}.pdb")
    if os.path.exists(outpath) and os.path.getsize(outpath) > 0:
        return outpath

    url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    with open(outpath, "wb") as f:
        f.write(r.content)
    return outpath



def _download_pdbe_updated_cif(pdb_id: str, outdir: str) -> str:
    """Download the PDBe SIFTS-enriched mmCIF when available."""
    pdb_id = (pdb_id or "").strip().lower()
    if not pdb_id:
        return ""
    os.makedirs(outdir, exist_ok=True)
    outpath = os.path.join(outdir, f"{pdb_id}_updated.cif")
    if os.path.exists(outpath) and os.path.getsize(outpath) > 1000:
        return outpath
    urls = [
        f"https://www.ebi.ac.uk/pdbe/entry-files/download/{pdb_id}_updated.cif",
        f"https://files.rcsb.org/download/{pdb_id.upper()}.cif",
    ]
    for url in urls:
        try:
            r = requests.get(url, timeout=30)
            if r.ok and r.text and len(r.text) > 1000:
                with open(outpath, "w", encoding="utf-8") as f:
                    f.write(r.text)
                return outpath
        except Exception:
            pass
    return ""


def _parse_atom_site_sifts_map_from_cif(cif_path: str, uniprot_acc: str = None, chain: str = None):
    """
    Parse SIFTS-enriched mmCIF atom_site fields into {author_resseq: UniProt_pos}.
    Expected fields in PDBe updated mmCIF include variants of:
      _atom_site.pdbx_sifts_xref_db_acc
      _atom_site.pdbx_sifts_xref_db_num
    plus author residue/chain fields. Falls back empty if those columns are absent.
    """
    uniprot_acc = (uniprot_acc or state.get("acc") or "").strip().upper().split("-")[0]
    chain = (chain or "").strip().upper()
    if not cif_path or not os.path.exists(cif_path):
        return {}

    try:
        lines = open(cif_path, "r", encoding="utf-8", errors="ignore").read().splitlines()
    except Exception:
        return {}

    out = {}
    i = 0
    n = len(lines)
    while i < n:
        if lines[i].strip() != "loop_":
            i += 1
            continue
        i += 1
        headers = []
        while i < n and lines[i].strip().startswith("_"):
            headers.append(lines[i].strip())
            i += 1
        if not headers or not any(h.startswith("_atom_site.") for h in headers):
            continue

        hmap = {h: idx for idx, h in enumerate(headers)}
        def _find(*suffixes):
            for s in suffixes:
                for h, idx in hmap.items():
                    if h.lower().endswith(s.lower()):
                        return idx
            return None

        idx_acc = _find("pdbx_sifts_xref_db_acc", "pdbx_sifts_xref_db_accession", "pdbx_sifts_xref_db_name")
        idx_num = _find("pdbx_sifts_xref_db_num", "pdbx_sifts_xref_db_res", "pdbx_sifts_xref_db_residue_number")
        idx_auth_seq = _find("auth_seq_id")
        idx_auth_chain = _find("auth_asym_id")
        idx_label_seq = _find("label_seq_id")
        idx_label_chain = _find("label_asym_id")
        if idx_num is None or (idx_auth_seq is None and idx_label_seq is None):
            # Atom-site loop exists but does not carry SIFTS residue numbers.
            # Continue scanning in case another atom_site loop has them.
            while i < n and not lines[i].strip().startswith(("loop_", "_")):
                i += 1
            continue

        import shlex
        while i < n:
            raw = lines[i].strip()
            if not raw or raw.startswith("#"):
                i += 1
                break
            if raw == "loop_" or raw.startswith("_"):
                break
            try:
                vals = shlex.split(raw)
            except Exception:
                vals = raw.split()
            i += 1
            if len(vals) < len(headers):
                continue
            try:
                acc_val = vals[idx_acc].upper().split("-")[0] if idx_acc is not None else uniprot_acc
                if uniprot_acc and acc_val not in {uniprot_acc, "?", ".", ""}:
                    continue
                ch_val = ""
                if idx_auth_chain is not None:
                    ch_val = vals[idx_auth_chain].strip().upper()
                elif idx_label_chain is not None:
                    ch_val = vals[idx_label_chain].strip().upper()
                if chain and ch_val and ch_val != chain:
                    continue
                pdb_res = vals[idx_auth_seq if idx_auth_seq is not None else idx_label_seq]
                uni_res = vals[idx_num]
                if pdb_res in {"?", "."} or uni_res in {"?", "."}:
                    continue
                pdb_res = int(float(pdb_res))
                uni_res = int(float(uni_res))
                out.setdefault(pdb_res, uni_res)
            except Exception:
                continue
    return out

def _fetch_pdbe_uniprot_author_map(pdb_id: str, uniprot_acc: str = None, chain: str = None):
    """
    Return dict mapping author/PDB residue numbers -> UniProt residue numbers for one PDB chain.
    Uses PDBe SIFTS UniProt mappings. Falls back safely if PDBe is unavailable.
    """
    pdb_id = (pdb_id or "").strip().lower()
    chain = (chain or "").strip().upper()[:1] if chain else None
    uniprot_acc = (uniprot_acc or state.get("acc") or "").strip().upper()
    if not pdb_id or not uniprot_acc:
        return {}

    cache = state.setdefault("pdbe_uniprot_maps", {})
    cache_key = (pdb_id.upper(), uniprot_acc, chain or "")
    if cache_key in cache:
        return cache[cache_key]

    out = {}
    try:
        url = f"https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{pdb_id}"
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        payload = r.json()
        data = payload.get(pdb_id, {}) or payload.get(pdb_id.lower(), {}) or payload.get(pdb_id.upper(), {}) or {}
        uni_block = data.get("UniProt", {}) or data.get("uniprot", {}) or {}

        # Prefer the requested accession, but allow isoform/key variants if needed.
        candidate_keys = []
        if uniprot_acc in uni_block:
            candidate_keys.append(uniprot_acc)
        candidate_keys += [k for k in uni_block.keys() if str(k).upper().split("-")[0] == uniprot_acc.split("-")[0] and k not in candidate_keys]
        if not candidate_keys:
            candidate_keys = list(uni_block.keys())

        for uk in candidate_keys:
            for m in (uni_block.get(uk, {}) or {}).get("mappings", []) or []:
                ch = (m.get("chain_id") or m.get("author_chain_id") or m.get("struct_asym_id") or "").strip().upper()[:1]
                if chain and ch and ch != chain:
                    continue

                unp_start = m.get("unp_start") or m.get("uniprot_start")
                unp_end = m.get("unp_end") or m.get("uniprot_end")
                start = m.get("start", {}) or {}
                end = m.get("end", {}) or {}
                pdb_start = start.get("author_residue_number", start.get("residue_number"))
                pdb_end = end.get("author_residue_number", end.get("residue_number"))
                try:
                    unp_start, unp_end = int(unp_start), int(unp_end)
                    pdb_start, pdb_end = int(pdb_start), int(pdb_end)
                except Exception:
                    continue

                step_p = 1 if pdb_end >= pdb_start else -1
                step_u = 1 if unp_end >= unp_start else -1
                n = min(abs(pdb_end - pdb_start), abs(unp_end - unp_start)) + 1
                for i in range(n):
                    out[pdb_start + i * step_p] = unp_start + i * step_u
    except Exception:
        out = {}

    # If the endpoint does not provide a usable residue-level map, try the SIFTS-enriched mmCIF.
    if not out:
        try:
            cif_path = _download_pdbe_updated_cif(pdb_id, state.get("workdir", tempfile.gettempdir()))
            out = _parse_atom_site_sifts_map_from_cif(cif_path, uniprot_acc=uniprot_acc, chain=chain)
        except Exception:
            out = {}

    cache[cache_key] = out
    return out

def _infer_chain_residue_range_from_pdb(pdb_path: str, chain: str = None):
    chain = (chain or "").strip().upper()[:1] if chain else None
    vals = []
    try:
        with open(pdb_path, "r") as fh:
            for line in fh:
                if not line.startswith(("ATOM", "HETATM")):
                    continue
                ch = line[21].strip().upper()
                if chain and ch != chain:
                    continue
                try:
                    vals.append(int(line[22:26].strip()))
                except Exception:
                    pass
    except Exception:
        pass
    if vals:
        return min(vals), max(vals)
    return None


def _build_structure_position_maps(pdb_id: str, chain: str = None, uni_range=None, pdb_path: str = None):
    """
    Shared structure mapping helper used by PTM/variant and Bio2Byte rendering.

    Returns:
      pdb_to_uniprot: {PDB author residue number -> UniProt residue number}
      uniprot_to_pdb: {UniProt residue number -> [PDB author residue numbers]}

    Priority:
      1. PDBe/SIFTS residue-level mapping.
      2. Chain-range offset mapping from UniProt PDB xrefs + actual PDB residue range.
      3. Direct numbering fallback.
    """
    pdb_to_uniprot = _fetch_pdbe_uniprot_author_map(pdb_id, state.get("acc"), chain) if pdb_id else {}

    if not pdb_to_uniprot and uni_range and pdb_path:
        try:
            u0, u1 = int(uni_range[0]), int(uni_range[1])
            pdb_range = _infer_chain_residue_range_from_pdb(pdb_path, chain)
            if pdb_range:
                p0, p1 = int(pdb_range[0]), int(pdb_range[1])
                n = min(abs(p1 - p0), abs(u1 - u0)) + 1
                step_p = 1 if p1 >= p0 else -1
                step_u = 1 if u1 >= u0 else -1
                pdb_to_uniprot = {p0 + i * step_p: u0 + i * step_u for i in range(n)}
        except Exception:
            pdb_to_uniprot = {}

    uniprot_to_pdb = {}
    for pdb_resi, uni_pos in (pdb_to_uniprot or {}).items():
        try:
            uniprot_to_pdb.setdefault(int(uni_pos), []).append(int(pdb_resi))
        except Exception:
            continue

    return pdb_to_uniprot or {}, uniprot_to_pdb or {}

def _build_nglview_from_pdbfile(pdb_path: str):
    v = nv.show_file(pdb_path)
    v.camera = "orthographic"
    try:
        v._remote_call("setParameters", target="stage", kwargs={"tooltip": True})
    except Exception:
        pass
    return v

def _style_base_ngl(view, chain=None):
    try:
        view.clear_representations()
    except Exception:
        pass
    chain = (chain or "").strip()
    if chain:
        chain = chain.upper()
        # NGL selection syntax is more reliable as :CHAIN than "chain A".
        sel = f"protein and :{chain}"
    else:
        sel = "protein"
    # Default context: everything is grey, then PTM/variant residues are overlaid.
    view.add_representation("cartoon", selection=sel, color="lightgrey")

def _collect_ptms_from_table(ptm_table):
    if ptm_table is None:
        return []
    pos_col = next((c for c in ["position","modpos","Position","MODPOS"] if c in ptm_table.columns), None)
    res_col = next((c for c in ["residue","modres","Residue","MODRES"] if c in ptm_table.columns), None)
    if pos_col is None or res_col is None:
        return []
    out = []
    for _, row in ptm_table[[pos_col, res_col]].dropna().iterrows():
        try:
            pos = int(row[pos_col])
        except Exception:
            continue
        res = str(row[res_col]).strip().upper()
        out.append((pos, res))
    return out

def _collect_variant_positions(variants_df):
    if variants_df is None or len(variants_df) == 0:
        return []
    if "position" not in variants_df.columns:
        return []
    out = []
    for v in variants_df["position"].dropna().tolist():
        try:
            out.append(int(v))
        except Exception:
            continue
    return sorted(set(out))

def _site_cartoon_selection(resi: int, chain: str = None, pad: int = 1):
    """
    NGL residue selection syntax. Using e.g. 899-901:A is more reliable
    than boolean strings for cartoon overlays in nglview.
    """
    resi = int(resi)
    lo, hi = max(1, resi - int(pad)), resi + int(pad)
    ch = (chain or "").strip().upper()[:1]
    return f"{lo}-{hi}:{ch}" if ch else f"{lo}-{hi}"

def _site_atom_selection(resi: int, chain: str = None):
    resi = int(resi)
    ch = (chain or "").strip().upper()[:1]
    return f"{resi}:{ch}" if ch else f"{resi}"

def _highlight_sites_ngl(view, ptm_table, variants_df=None, chain=None, mode="both", site_style="stick", pdb_to_uniprot=None, component=None, color_mode="status", residue_color_map=None):
    # color_mode="status" uses PTM/variant colors; color_mode="bfactor" inherits the active Bio2Byte/property B-factor colour.
    """
    Highlight UniProt-indexed PTMs/variants on AF or PDB structures using visible atom-level overlays.

    Baseline/cartoon coloring is handled by the caller. This function only adds site overlays,
    because NGL cartoon recoloring of tiny residue patches is unreliable in Voila/nglview.

    mode:
      ptm      = PTM residues only
      mut      = variant residues only
      both     = PTM OR variant residues
      overlap  = only positions that are both PTM and variant
    """
    ptms = _collect_ptms_from_table(ptm_table)
    muts = set(_collect_variant_positions(variants_df))
    chain = (chain or "").strip().upper()[:1]

    ptm_pos_to_res = {int(pos): str(res).strip().upper() for pos, res in ptms if pos is not None}
    ptm_set = set(ptm_pos_to_res.keys())
    mut_set = set(muts)

    m = (mode or "both").lower()
    if m == "ptm":
        uni_positions = sorted(ptm_set)
    elif m in {"mut", "mutation", "variant"}:
        uni_positions = sorted(mut_set)
    elif m in {"overlap", "ptm+mutation", "ptm_variant"}:
        uni_positions = sorted(ptm_set & mut_set)
    else:
        uni_positions = sorted(ptm_set | mut_set)
    if not uni_positions:
        return

    if pdb_to_uniprot:
        uniprot_to_pdb = {}
        for pdb_resi, uni_pos in pdb_to_uniprot.items():
            try:
                uniprot_to_pdb.setdefault(int(uni_pos), []).append(int(pdb_resi))
            except Exception:
                continue
    else:
        uniprot_to_pdb = {int(p): [int(p)] for p in uni_positions}

    rep = (site_style or "stick").lower()
    if rep in {"cartoon", "cartoon only", "none", "cartoon_color"}:
        rep = "stick"

    def _normalise_ngl_color(c):
        """NGL accepts CSS-style hex colours more reliably than py3Dmol 0xRRGGBB."""
        if c is None:
            return None
        c = str(c).strip()
        if c.startswith("0x") and len(c) == 8:
            return "#" + c[2:]
        return c

    def _add(rep_name, sel, color, pdb_resi=None):
        kwargs = {"selection": sel}
        explicit_color = None
        if residue_color_map is not None and pdb_resi is not None:
            try:
                explicit_color = _normalise_ngl_color(residue_color_map.get(int(pdb_resi)))
            except Exception:
                explicit_color = None

        cmode = (color_mode or "status").lower()
        if cmode in {"marker", "representation_only", "neutral", "grey", "gray"}:
            # Marker-only mode: use a neutral site marker so Bio2Byte/cartoon colours remain the data layer.
            kwargs["color_scheme"] = "uniform"
            kwargs["color_value"] = "#7a7a7a"
        elif cmode in {"none", "element"}:
            # Rare fallback: element-coloured overlay.
            pass
        elif explicit_color:
            kwargs["color_scheme"] = "uniform"
            kwargs["color_value"] = explicit_color
        elif cmode in {"bfactor", "property", "bio2byte"}:
            kwargs["color_scheme"] = "bfactor"
        else:
            kwargs["color_scheme"] = "uniform"
            kwargs["color_value"] = _normalise_ngl_color(color)
        if component is not None:
            kwargs["component"] = component
        if rep_name == "spacefill":
            kwargs["radius"] = 0.9
        view.add_representation(rep_name, **kwargs)

    for uni_pos in uni_positions:
        in_ptm = uni_pos in ptm_set
        in_mut = uni_pos in mut_set

        if m == "ptm" and not in_ptm:
            continue
        if m in {"mut", "mutation", "variant"} and not in_mut:
            continue
        if m in {"overlap", "ptm+mutation", "ptm_variant"} and not (in_ptm and in_mut):
            continue

        if in_ptm and in_mut:
            color = "purple"
        elif in_mut:
            color = "red"
        else:
            color = PTM_COLOR.get(ptm_pos_to_res.get(uni_pos, ""), "orange")

        for pdb_resi in uniprot_to_pdb.get(int(uni_pos), []):
            atom_sel = _site_atom_selection(int(pdb_resi), chain=chain)
            if rep == "sphere":
                _add("spacefill", atom_sel, color, pdb_resi=pdb_resi)
            elif rep == "ballstick":
                _add("ball+stick", atom_sel, color, pdb_resi=pdb_resi)
            else:
                _add("licorice", atom_sel, color, pdb_resi=pdb_resi)



def _render_structure_sites_py3dmol(pdb_path: str, ptm_table=None, variants_df=None, chain=None, mode="both", site_style="cartoon", pdb_to_uniprot=None, width=950, height=620):
    """
    Reliable Tab 3 renderer using py3Dmol.
    Base protein is grey cartoon; PTM/variant residues are recolored on cartoon.
    Optional stick/sphere overlays are added on top.
    """
    with open(pdb_path, "r") as f:
        pdb_txt = f.read()

    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb_txt, "pdb")

    ch = (chain or "").strip().upper()[:1]
    base_sel = {"chain": ch} if ch else {}
    view.setStyle(base_sel, {"cartoon": {"color": "lightgrey"}})

    ptms = _collect_ptms_from_table(ptm_table)
    muts = set(_collect_variant_positions(variants_df))
    ptm_pos_to_res = {int(pos): str(res).strip().upper() for pos, res in ptms if pos is not None}

    if mode == "ptm":
        uni_positions = sorted(ptm_pos_to_res.keys())
    elif mode == "mut":
        uni_positions = sorted(muts)
    else:
        uni_positions = sorted(set(ptm_pos_to_res.keys()) | set(muts))

    if pdb_to_uniprot:
        uniprot_to_pdb = {}
        for pdb_resi, uni_pos in pdb_to_uniprot.items():
            try:
                uniprot_to_pdb.setdefault(int(uni_pos), []).append(int(pdb_resi))
            except Exception:
                continue
    else:
        uniprot_to_pdb = {int(p): [int(p)] for p in uni_positions}

    rep = (site_style or "cartoon").lower()
    if rep in {"none", "cartoon only", "cartoon_color"}:
        rep = "cartoon"

    for uni_pos in uni_positions:
        in_ptm = uni_pos in ptm_pos_to_res
        in_mut = uni_pos in muts
        if mode == "ptm" and not in_ptm:
            continue
        if mode == "mut" and not in_mut:
            continue

        if mode == "ptm":
            color = PTM_COLOR.get(ptm_pos_to_res.get(uni_pos, ""), "orange")
        elif mode == "mut":
            color = "red"
        else:
            if in_ptm and in_mut:
                color = "purple"
            elif in_mut:
                color = "red"
            else:
                color = PTM_COLOR.get(ptm_pos_to_res.get(uni_pos, ""), "orange")

        for pdb_resi in uniprot_to_pdb.get(int(uni_pos), []):
            # A ±1 residue cartoon patch makes single positions visible on cartoon.
            lo, hi = max(1, int(pdb_resi) - 1), int(pdb_resi) + 1
            patch_sel = {"resi": f"{lo}-{hi}"}
            atom_sel = {"resi": str(int(pdb_resi))}
            if ch:
                patch_sel["chain"] = ch
                atom_sel["chain"] = ch

            view.setStyle(patch_sel, {"cartoon": {"color": color}})
            if rep == "stick":
                view.addStyle(atom_sel, {"stick": {"color": color, "radius": 0.28}})
            elif rep == "sphere":
                view.addStyle(atom_sel, {"sphere": {"color": color, "radius": 1.0}})

    view.setBackgroundColor("white")
    view.zoomTo(base_sel if base_sel else {})
    view.render()
    return view

def on_fetch_af(_):
    try:
        with out_3d:
            clear_output()
            if not state.get("acc"):
                print("Set a UniProt accession first (header).")
                return
            print("Downloading AlphaFold model...")
            af_path = download_alphafold_pdb(state["acc"], out_dir=state["workdir"])
            state["af_path"] = af_path
            print("✅ Saved:", af_path)
    except Exception as e:
        with out_3d:
            print("❌ Failed to fetch AlphaFold:", e)

def on_show_3d(_):
    try:
        with out_3d:
            clear_output()

            ptm_table = state.get("ptm_table")
            use_py3dmol = (tab3_viewer_mode.value == "py3dmol")

            # AlphaFold branch
            if structure_source.value == "af":
                af_path = state.get("af_path")
                if not af_path or not os.path.exists(af_path):
                    print("No AlphaFold model found. Click 'Fetch AlphaFold' first.")
                    return

                if use_py3dmol:
                    v = _render_structure_sites_py3dmol(
                        af_path,
                        ptm_table=ptm_table,
                        variants_df=state.get("variants_df"),
                        chain="A",
                        mode=map_mode.value,
                        site_style=site_rep_mode.value,
                    )
                else:
                    v = _build_nglview_from_pdbfile(af_path)
                    _style_base_ngl(v, chain="A")
                    _highlight_sites_ngl(
                        v,
                        ptm_table,
                        state.get("variants_df"),
                        chain="A",
                        mode=map_mode.value,
                        site_style=site_rep_mode.value,
                    )
                    try:
                        v.center(selection=":A")
                    except Exception:
                        pass

                state["tab3_last_view"] = v
                state["tab3_last_pdb_path"] = af_path
                state["tab3_last_pdb_id"] = None
                state["tab3_last_chain"] = "A"
                display(v)
                return

            # PDB branch
            pdb = pdb_input.value.strip().upper()
            if not pdb:
                print("Enter a PDB ID.")
                return

            ch = chain_input.value.strip() or None
            if ch:
                ch = ch[0].upper()

            pdb_path = _download_pdb_to_workdir(pdb, state["workdir"])
            pdb_to_uniprot, _ = _build_structure_position_maps(pdb, chain=ch, uni_range=None, pdb_path=pdb_path)

            if use_py3dmol:
                v = _render_structure_sites_py3dmol(
                    pdb_path,
                    ptm_table=ptm_table,
                    variants_df=state.get("variants_df"),
                    chain=ch,
                    mode=map_mode.value,
                    site_style=site_rep_mode.value,
                    pdb_to_uniprot=pdb_to_uniprot,
                )
            else:
                v = _build_nglview_from_pdbfile(pdb_path)
                _style_base_ngl(v, chain=ch)
                _highlight_sites_ngl(
                    v,
                    ptm_table,
                    state.get("variants_df"),
                    chain=ch,
                    mode=map_mode.value,
                    site_style=site_rep_mode.value,
                    pdb_to_uniprot=pdb_to_uniprot,
                )
                try:
                    if ch:
                        v.center(selection=f":{ch}")
                    else:
                        v.center()
                except Exception:
                    pass

            state["tab3_last_view"] = v
            state["tab3_last_pdb_path"] = pdb_path
            state["tab3_last_pdb_id"] = pdb
            state["tab3_last_chain"] = ch
            display(v)

    except Exception as e:
        with out_3d:
            print("❌ Error:", e)

def on_export_tab3(_):
    _export_current_view(
        f"tab3_{state.get('acc','protein')}_{structure_source.value}",
        "tab3_last_view",
        "tab3_last_pdb_path",
        "tab3_last_pdb_id",
        out=out_3d,
    )

btn_fetch_af.on_click(on_fetch_af)
btn_show_3d.on_click(on_show_3d)
btn_export_tab3.on_click(on_export_tab3)

pdb_controls = w.VBox([
    w.HBox([pdb_input, chain_input]),
    w.HBox([pdb_dd, chain_dd]),
    range_lbl,
])

def _sync_tab3_visibility(*_):
    with out_3d:
        clear_output()

    if structure_source.value == "af":
        pdb_input.value = ""
        chain_input.value = ""
        try: pdb_dd.value = ""
        except Exception: pass
        try: chain_dd.value = ""
        except Exception: pass

        pdb_controls.layout.display = "none"
        btn_fetch_af.layout.display = ""
        with out_3d:
            print("AlphaFold selected. Click 'Fetch AlphaFold' (then 'Show 3D').")
    else:
        pdb_controls.layout.display = ""
        btn_fetch_af.layout.display = "none"
        with out_3d:
            print("PDB selected. Enter a PDB ID + (optional) chain, or pick from the UniProt dropdowns.")

structure_source.observe(_sync_tab3_visibility, names="value")
_sync_tab3_visibility()

tab3 = w.VBox([
    w.HTML("<h3>3D Structure Viewer</h3>"),
    w.HBox([structure_source, btn_fetch_af, btn_show_3d, btn_export_tab3, map_mode, site_rep_mode]),
    w.HBox([tab3_viewer_mode]),
    pdb_controls,
    out_3d
])

# Initialize Tab 3 dropdowns (safe)
try:
    _refresh_pdb_dropdowns()
except Exception:
    pass

# ----------------------------
# TAB 4: Bio2Byte predictions (on-demand)
# ----------------------------
out_b2b = w.Output(layout={"border":"1px solid #ddd","padding":"6px"})
b2b_legend = w.HTML("")  # dynamic Bio2Byte colour legend
btn_fetch_seq = w.Button(description="Fetch sequence", button_style="warning")
btn_run_b2b = w.Button(description="Run predictions", button_style="danger")
btn_show_b2b = w.Button(description="Show prediction table", button_style="success")
btn_show_b2b3d = w.Button(description="3D panel (Bio2Byte colors)", button_style="")
btn_export_b2b = w.Button(description="Export Bio2Byte view/structure", button_style="", tooltip="Export last Bio2Byte view and B-factor-colored PDB/CIF when available")

b2b_metric = w.Dropdown(description="Color by:", options=[], layout=w.Layout(width="320px"))

b2b_site_overlay = w.Dropdown(
    description="Show sites:",
    options=[
        ("No site overlay", "none"),
        ("PTMs", "ptm"),
        ("Variants", "mut"),
        ("PTM + variant overlap", "overlap"),
        ("All PTM/variant sites", "both"),
    ],
    value="none",
    layout=w.Layout(width="320px")
)

b2b_site_style = w.Dropdown(
    description="Site style:",
    options=[("Sticks/licorice", "stick"), ("Sphere", "sphere"), ("Ball+stick", "ballstick")],
    value="stick",
    layout=w.Layout(width="300px")
)

viewer_mode = w.ToggleButtons(
    options=[("Single interactive (NGL)", "ngl"), ("4-panel (py3Dmol)", "py3dmol")],
    value="ngl",
    description="Viewer:",
    layout=w.Layout(margin="0 0 0 10px")
)

# structure source
b2b_source = w.ToggleButtons(
    options=[("AlphaFold", "af"), ("PDB", "pdb")],
    value="af",
    description="Structure:",
    layout=w.Layout(width="360px")
)

# PDB entry + dropdowns
b2b_pdb_text  = w.Text(description="PDB ID:", placeholder="e.g. 6CER", layout=w.Layout(width="220px"))
b2b_chain_text = w.Text(description="Chain:", placeholder="A", layout=w.Layout(width="160px"))

b2b_pdb_dropdown   = w.Dropdown(description="UniProt PDB:", options=[""], layout=w.Layout(width="320px"))
b2b_chain_dropdown = w.Dropdown(description="Chain/range:", options=[("", "")], layout=w.Layout(width="380px"))

# ---- FIXED: Tab 4 dropdown population (show ALL chains, ranges if available) ----
def _refresh_b2b_pdb_dropdowns():
    """
    Populate Tab 4 PDB + chain/range dropdowns from state['uniprot_pdb_refs'].

    Shows ALL chains mentioned by UniProt.
    If range is known -> label "A (UniProt 10–220)" value "A|10|220"
    If range is n/a    -> label "A (range n/a)"       value "A||"
    """
    refs = state.get("uniprot_pdb_refs") or []

    pdb_to_chains = {}  # pid -> { chain -> (u0,u1) or None }

    for r in refs:
        pid = (r.get("pdb_id") or "").strip().upper()
        if not pid:
            continue
        chain_map = pdb_to_chains.setdefault(pid, {})

        cr = r.get("chain_ranges") or {}
        for ch, rng in cr.items():
            if not ch:
                continue
            ch = ch.strip().upper()[:1]
            if rng and isinstance(rng, (tuple, list)) and len(rng) == 2:
                try:
                    chain_map[ch] = (int(rng[0]), int(rng[1]))
                except Exception:
                    chain_map.setdefault(ch, None)
            else:
                chain_map.setdefault(ch, None)

        # Also include any chain letters in raw_chains even if range not parsed
        raw = (r.get("raw_chains") or "").strip()
        if raw:
            for part in [p.strip() for p in raw.split(",") if p.strip()]:
                if "=" not in part:
                    continue
                lhs = part.split("=", 1)[0].strip()
                for ch in re.split(r"[\/\s]+", lhs):
                    ch = (ch or "").strip().upper()
                    if not ch:
                        continue
                    ch = ch[:1]
                    chain_map.setdefault(ch, None)

    state["b2b_pdb_chain_map"] = pdb_to_chains

    pdb_ids = sorted(pdb_to_chains.keys())
    b2b_pdb_dropdown.options = [""] + pdb_ids

    if b2b_pdb_dropdown.value not in ([""] + pdb_ids):
        b2b_pdb_dropdown.value = ""

    _b2b_on_pdb_pick()

def _b2b_on_pdb_pick(change=None):
    pid = (b2b_pdb_dropdown.value or "").strip().upper()
    pdb_map = state.get("b2b_pdb_chain_map") or {}

    if not pid or pid not in pdb_map:
        b2b_chain_dropdown.options = [("", "")]
        b2b_chain_dropdown.value = ""
        return

    chain_map = pdb_map[pid]  # { 'A': (u0,u1) or None }

    opts = [("", "")]
    for ch in sorted(chain_map.keys()):
        rng = chain_map[ch]
        if rng and len(rng) == 2:
            u0, u1 = int(rng[0]), int(rng[1])
            opts.append((f"{ch} (UniProt {u0}–{u1})", f"{ch}|{u0}|{u1}"))
        else:
            opts.append((f"{ch} (range n/a)", f"{ch}||"))

    b2b_chain_dropdown.options = opts
    b2b_chain_dropdown.value = ""

def _b2b_on_chain_pick(change=None):
    val = b2b_chain_dropdown.value or ""
    if "|" in val:
        parts = (val.split("|") + ["", ""])[:3]
        ch = (parts[0] or "").strip().upper()[:1]
        if ch:
            b2b_chain_text.value = ch
        if b2b_pdb_dropdown.value:
            b2b_pdb_text.value = b2b_pdb_dropdown.value

b2b_pdb_dropdown.observe(_b2b_on_pdb_pick, names="value")
b2b_chain_dropdown.observe(_b2b_on_chain_pick, names="value")

# ---- end dropdown plumbing ----

import colorsys
import py3Dmol
import pandas as pd

def download_rcsb_pdb(pdbid: str, out_dir: str) -> str:
    pdbid = pdbid.strip().upper()
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{pdbid}.pdb")
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
        return out_path
    url = f"https://files.rcsb.org/download/{pdbid}.pdb"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)
    return out_path

def pseudocolor(minval, maxval, val):
    minval, maxval = float(minval), float(maxval)
    if maxval == minval:
        h = 120.0
    else:
        h = (float(maxval - val) / (maxval - minval)) * 120.0
    r, g, b = colorsys.hsv_to_rgb(h/360.0, 1.0, 1.0)
    rgb = tuple(int(255*x) for x in (r, g, b))
    return "0x%02x%02x%02x" % rgb

B2B_THRESHOLDS = {
    "backbone": [
        {"min": None, "max": 0.69, "label": "flexible"},
        {"min": 0.69, "max": 0.80, "label": "context dependent"},
        {"min": 0.80, "max": 1.00, "label": "rigid"},
        {"min": 1.00, "max": None, "label": "membrane spanning"},
    ],
    "disoMine": [
        {"min": None, "max": 0.5, "label": "ordered"},
        {"min": 0.5, "max": None, "label": "disordered"},
    ],
    "earlyFolding": [
        {"min": None, "max": 0.169, "label": "not early folding"},
        {"min": 0.169, "max": None, "label": "early folding"},
    ],
}

def _b2b_interpretation(prop, val):
    """Return the interpretation-band label for a value (e.g. 'rigid'), or None."""
    bands = B2B_THRESHOLDS.get(prop)
    if not bands or val is None:
        return None
    try:
        v = float(val)
    except Exception:
        return None
    for b in bands:
        lo, hi = b["min"], b["max"]
        if (lo is None or v >= lo) and (hi is None or v < hi):
            return b["label"]
    return None

def render_b2b_legend(property_name, vmin, vmax, n=48):
    """Colour-scale legend matching pseudocolor() (low=green -> high=red), with
    interpretation bands + cut-off markers for properties in B2B_THRESHOLDS."""
    try:
        vmin = float(vmin); vmax = float(vmax)
    except Exception:
        return ""
    if vmax <= vmin:
        vmax = vmin + 1e-6
    stops = []
    for k in range(n + 1):
        v = vmin + (vmax - vmin) * (k / n)
        hexc = "#" + pseudocolor(vmin, vmax, v).replace("0x", "")
        stops.append(f"{hexc} {100.0 * k / n:.1f}%")
    grad = "linear-gradient(to right, " + ", ".join(stops) + ")"

    def pct(x):
        return max(0.0, min(100.0, (x - vmin) / (vmax - vmin) * 100.0))

    bands = B2B_THRESHOLDS.get(property_name)
    markers_html = top_html = labels_html = ""
    if bands:
        for t in sorted({b["max"] for b in bands if b["max"] is not None}):
            if vmin <= t <= vmax:
                p = pct(t)
                markers_html += (f"<div style='position:absolute;left:{p:.2f}%;top:0;height:14px;"
                                 f"width:0;border-left:2px solid #222;'></div>")
                top_html += (f"<span style='position:absolute;left:{p:.2f}%;top:0;transform:translateX(-50%);"
                             f"font-size:10px;color:#222;'>{t:g}</span>")
        for b in bands:
            lo = vmin if b["min"] is None else b["min"]
            hi = vmax if b["max"] is None else b["max"]
            a, z = max(lo, vmin), min(hi, vmax)
            if z - a > (vmax - vmin) * 0.04:
                c = pct((a + z) / 2.0)
                labels_html += (f"<span style='position:absolute;left:{c:.2f}%;transform:translateX(-50%);"
                                f"white-space:nowrap;'>{b['label']}</span>")
    # values (min, cut-offs, max) go ABOVE the bar; interpretation labels go BELOW it
    top_html = (f"<span style='position:absolute;left:0;top:0;'>{vmin:.3g}</span>"
                f"<span style='position:absolute;right:0;top:0;'>{vmax:.3g}</span>" + top_html)
    return (
        "<div style='margin:6px 0 24px 0;font-size:12px;max-width:600px;'>"
        f"<div style='font-weight:600;margin-bottom:4px;'>{property_name} \u2014 colour scale</div>"
        f"<div style='position:relative;height:16px;color:#333;margin-bottom:4px;'>{top_html}</div>"
        "<div style='position:relative;height:16px;'>"
        f"<div style='height:14px;border:1px solid #ccc;border-radius:3px;background:{grad};'></div>"
        f"{markers_html}</div>"
        f"<div style='position:relative;height:16px;margin-top:3px;color:#333;'>{labels_html}</div>"
        "</div>"
    )

def _b2b_col_range(dyn, col):
    try:
        vals = pd.to_numeric(dyn[col], errors="coerce").dropna()
        if vals.empty:
            return None
        return float(vals.min()), float(vals.max())
    except Exception:
        return None

def _update_b2b_legend(*_):
    dyn = state.get("dynamic_properties")
    if dyn is None:
        b2b_legend.value = ""
        return
    if viewer_mode.value == "ngl":
        col = b2b_metric.value
        rng = _b2b_col_range(dyn, col) if col else None
        b2b_legend.value = render_b2b_legend(col, rng[0], rng[1]) if rng else ""
    else:
        parts = []
        for col in ["backbone", "disoMine", "earlyFolding"]:
            if col in dyn.columns:
                rng = _b2b_col_range(dyn, col)
                if rng:
                    parts.append(render_b2b_legend(col, rng[0], rng[1]))
        parts.append("<div style='font-size:12px;color:#555;max-width:520px;'>"
                     "Top-left panel colours the structure B-factor / pLDDT (blue\u2192white\u2192red, 0\u2013100).</div>")
        b2b_legend.value = "".join(parts)

def _update_rin_legend(*_):
    dyn = state.get("dynamic_properties")
    if rin_color_mode.value != "b2b" or dyn is None or not rin_b2b_metric.value:
        rin_legend.value = ""
        return
    rng = _b2b_col_range(dyn, rin_b2b_metric.value)
    rin_legend.value = render_b2b_legend(rin_b2b_metric.value, rng[0], rng[1]) if rng else ""

def remap_b2b_colors(df, pdb_to_uniprot=None):
    """Build py3Dmol residue-color maps. Keys are PDB residue numbers when a PDB->UniProt map is provided; otherwise UniProt/AF residue numbers."""
    base = {}
    min_BD, max_BD = float(df["backbone"].min()), float(df["backbone"].max())
    min_DO, max_DO = float(df["disoMine"].min()), float(df["disoMine"].max())
    min_EF, max_EF = float(df["earlyFolding"].min()), float(df["earlyFolding"].max())
    for i, row in df.iterrows():
        uni_pos = int(i) + 1
        base[uni_pos] = (
            pseudocolor(min_BD, max_BD, float(row["backbone"])),
            pseudocolor(min_EF, max_EF, float(row["earlyFolding"])),
            pseudocolor(min_DO, max_DO, float(row["disoMine"])),
        )

    if pdb_to_uniprot:
        BDcolor, EFcolor, DOcolor = {}, {}, {}
        for pdb_resi, uni_pos in pdb_to_uniprot.items():
            vals = base.get(int(uni_pos))
            if vals:
                BDcolor[int(pdb_resi)], EFcolor[int(pdb_resi)], DOcolor[int(pdb_resi)] = vals
        return BDcolor, EFcolor, DOcolor

    BDcolor = {p: v[0] for p, v in base.items()}
    EFcolor = {p: v[1] for p, v in base.items()}
    DOcolor = {p: v[2] for p, v in base.items()}
    return BDcolor, EFcolor, DOcolor

def display_b2b_4panel_py3dmol(dynamic_properties_df, pdb_path: str, chain=None, ptm_positions=None, pdb_to_uniprot=None):
    BDcolor, EFcolor, DOcolor = remap_b2b_colors(dynamic_properties_df, pdb_to_uniprot=pdb_to_uniprot)
    with open(pdb_path, "r") as f:
        pdb_txt = f.read()

    view = py3Dmol.view(viewergrid=(2, 2), linked=True, width=950, height=740)
    view.addModel(pdb_txt, "pdb")

    base_sel = {} if not chain else {"chain": chain}
    view.setStyle(base_sel, {"cartoon": {"colorscheme": {"prop": "b", "gradient": "rwb", "min": 0.0, "max": 100.0}}}, viewer=(0, 0))
    view.setStyle(base_sel, {"cartoon": {"colorscheme": {"prop": "resi", "map": BDcolor}}}, viewer=(0, 1))
    view.setStyle(base_sel, {"cartoon": {"colorscheme": {"prop": "resi", "map": DOcolor}}}, viewer=(1, 0))
    view.setStyle(base_sel, {"cartoon": {"colorscheme": {"prop": "resi", "map": EFcolor}}}, viewer=(1, 1))

    for panel in [(0,0), (0,1), (1,0), (1,1)]:
        view.setBackgroundColor("white", viewer=panel)

    view.zoomTo(base_sel if base_sel else {})
    view.render()

    # ---- residue hover: value + interpretation per property panel ----
    try:
        import json as _json
        from IPython.display import HTML as _HTML

        def _vmap(col):
            if col not in dynamic_properties_df.columns:
                return {}
            vals = pd.to_numeric(dynamic_properties_df[col], errors="coerce")
            uni_to_val = {i + 1: float(vals.iloc[i]) for i in range(len(vals)) if pd.notna(vals.iloc[i])}
            m = {}
            if pdb_to_uniprot:
                for pr, up in pdb_to_uniprot.items():
                    if int(up) in uni_to_val:
                        vv = uni_to_val[int(up)]
                        it = _b2b_interpretation(col, vv)
                        m[str(int(pr))] = f"{vv:.3f}" + (f" ({it})" if it else "")
            else:
                for up, vv in uni_to_val.items():
                    it = _b2b_interpretation(col, vv)
                    m[str(int(up))] = f"{vv:.3f}" + (f" ({it})" if it else "")
            return m

        panels = {(0, 1): "backbone", (1, 0): "disoMine", (1, 1): "earlyFolding"}
        replacements = {}
        unhover = "function(atom,viewer){ if(atom.label){ viewer.removeLabel(atom.label); delete atom.label; } }"
        for panel, col in panels.items():
            vm = _vmap(col)
            htok, utok = f"__B2BHOVER_{col}__", f"__B2BUNHOVER_{col}__"
            view.setHoverable({}, True, htok, utok, viewer=panel)
            hover = ("function(atom,viewer,event,container){ if(!atom.label){ var m=" + _json.dumps(vm) + "; "
                     "var v=m[atom.resi]; var t=atom.resn+atom.resi+(v!==undefined?(' \\u00b7 " + col + " '+v):''); "
                     "atom.label=viewer.addLabel(t,{position:atom,backgroundColor:'black',backgroundOpacity:0.75,"
                     "fontColor:'white',fontSize:11}); } }")
            replacements['"' + htok + '"'] = hover
            replacements['"' + utok + '"'] = unhover

        html = view._make_html()
        for quoted, raw in replacements.items():
            html = html.replace(quoted, raw)
        return _HTML(html)
    except Exception:
        return view

def _b2b_residue_color_map(df, value_col: str, pdb_path: str, chain: str = None, uni_range=None, pdb_to_uniprot=None):
    """
    Return {PDB_resseq: color} using explicit min/max scaling for the chosen Bio2Byte metric.
    This avoids relying on original PDB B-factors or NGL bfactor color scaling.
    """
    vals = pd.to_numeric(df[value_col], errors="coerce")
    finite = vals.dropna()
    if finite.empty:
        return {}, None, None
    vmin, vmax = float(finite.min()), float(finite.max())
    uni_to_color = {int(i) + 1: pseudocolor(vmin, vmax, float(v)) for i, v in vals.items() if pd.notna(v)}

    # Preferred path: residue-level PDB -> UniProt mapping.
    if pdb_to_uniprot:
        out = {}
        for pdb_resi, uni_pos in pdb_to_uniprot.items():
            try:
                c = uni_to_color.get(int(uni_pos))
                if c is not None:
                    out[int(pdb_resi)] = c
            except Exception:
                continue
        return out, vmin, vmax

    # Fallback: UniProt chain range + actual PDB residue start.
    fallback_pdb_range = _infer_chain_residue_range_from_pdb(pdb_path, chain)
    if uni_range and fallback_pdb_range:
        try:
            u0, u1 = int(uni_range[0]), int(uni_range[1])
            p0, p1 = int(fallback_pdb_range[0]), int(fallback_pdb_range[1])
            n = min(abs(p1 - p0), abs(u1 - u0)) + 1
            step_p = 1 if p1 >= p0 else -1
            step_u = 1 if u1 >= u0 else -1
            out = {}
            for i in range(n):
                pdb_resi = p0 + i * step_p
                uni_pos = u0 + i * step_u
                if uni_pos in uni_to_color:
                    out[pdb_resi] = uni_to_color[uni_pos]
            return out, vmin, vmax
        except Exception:
            pass

    # AlphaFold/direct-numbering fallback.
    return dict(uni_to_color), vmin, vmax


def _write_b2b_scaled_bfactor_pdb(input_pdb: str, output_pdb: str, dyn_df, value_col: str, chain: str = None, uni_range=None, pdb_to_uniprot=None):
    """
    Write a temporary PDB where the B-factor column contains 0-100 scaled Bio2Byte values.
    This restores the interactive NGL viewer while keeping explicit selected-column min/max scaling.
    """
    resi_to_color, vmin, vmax = _b2b_residue_color_map(
        dyn_df,
        value_col=value_col,
        pdb_path=input_pdb,
        chain=chain,
        uni_range=uni_range,
        pdb_to_uniprot=pdb_to_uniprot,
    )

    vals = pd.to_numeric(dyn_df[value_col], errors="coerce")
    finite = vals.dropna()
    if finite.empty:
        raise ValueError(f"No numeric values found for {value_col}")
    vmin, vmax = float(finite.min()), float(finite.max())
    if vmax == vmin:
        uni_to_scaled = {int(i)+1: 50.0 for i, v in vals.items() if pd.notna(v)}
    else:
        uni_to_scaled = {int(i)+1: 100.0*(float(v)-vmin)/(vmax-vmin) for i, v in vals.items() if pd.notna(v)}

    if pdb_to_uniprot:
        pdb_to_scaled = {}
        for pdb_resi, uni_pos in pdb_to_uniprot.items():
            try:
                if int(uni_pos) in uni_to_scaled:
                    pdb_to_scaled[int(pdb_resi)] = uni_to_scaled[int(uni_pos)]
            except Exception:
                continue
    else:
        # Direct AF numbering or fallback map from UniProt range to PDB numbering.
        pdb_to_scaled = {}
        fallback_pdb_range = _infer_chain_residue_range_from_pdb(input_pdb, chain)
        if uni_range and fallback_pdb_range:
            try:
                u0, u1 = int(uni_range[0]), int(uni_range[1])
                p0, p1 = int(fallback_pdb_range[0]), int(fallback_pdb_range[1])
                n = min(abs(p1 - p0), abs(u1 - u0)) + 1
                step_p = 1 if p1 >= p0 else -1
                step_u = 1 if u1 >= u0 else -1
                for i in range(n):
                    pdb_resi = p0 + i * step_p
                    uni_pos = u0 + i * step_u
                    if uni_pos in uni_to_scaled:
                        pdb_to_scaled[pdb_resi] = uni_to_scaled[uni_pos]
            except Exception:
                pdb_to_scaled = dict(uni_to_scaled)
        else:
            pdb_to_scaled = dict(uni_to_scaled)

    ch = (chain or "").strip().upper()[:1]
    with open(input_pdb, "r") as fin, open(output_pdb, "w") as fout:
        for line in fin:
            if line.startswith(("ATOM  ", "HETATM")):
                try:
                    atom_chain = line[21].strip()
                    resi = int(line[22:26].strip())
                    if (not ch or atom_chain == ch) and resi in pdb_to_scaled:
                        b = max(0.0, min(100.0, float(pdb_to_scaled[resi])))
                    else:
                        b = 50.0
                    line = line[:60] + f"{b:6.2f}" + line[66:]
                except Exception:
                    pass
            fout.write(line)
    return output_pdb, vmin, vmax


def display_b2b_3D_ngl(dyn_df, pdb_path: str, value_col: str, chain: str = None, uni_range=None, pdb_to_uniprot=None, site_overlay="none", site_style="stick"):
    """
    Interactive Bio2Byte single-structure viewer using NGL/nglview.
    The selected Bio2Byte column is written to a temporary B-factor field with 0-100 scaling.
    """
    tmp_path = os.path.join(state.get("workdir", tempfile.gettempdir()), f"b2b_{os.path.basename(pdb_path)}_{value_col}.pdb")
    scaled_pdb, vmin, vmax = _write_b2b_scaled_bfactor_pdb(
        pdb_path,
        tmp_path,
        dyn_df,
        value_col=value_col,
        chain=chain,
        uni_range=uni_range,
        pdb_to_uniprot=pdb_to_uniprot,
    )
    v = _build_nglview_from_pdbfile(scaled_pdb)
    try:
        v.clear_representations()
    except Exception:
        pass
    sel = f":{chain}" if chain else "protein"
    try:
        v.add_representation("cartoon", selection=sel, color_scheme="bfactor",
                             color_scale="RdYlGn", color_reverse=True, color_domain=[0, 100])
    except Exception:
        v.add_representation("cartoon", selection=sel, color_scheme="bfactor")
    if (site_overlay or "none") != "none":
        try:
            # Tab 4: Bio2Byte colour remains on the cartoon; selected PTM/variant sites
            # are marked only by representation (sticks/spheres/ball+stick).
            _highlight_sites_ngl(
                v,
                state.get("ptm_table"),
                state.get("variants_df"),
                chain=chain,
                mode=site_overlay,
                site_style=site_style,
                pdb_to_uniprot=pdb_to_uniprot,
                color_mode="marker",
                residue_color_map=None,
            )
        except Exception as e:
            print("⚠️ Could not add PTM/variant overlays:", e)
    try:
        v.center(selection=sel)
    except Exception:
        pass
    state["tab4_last_colored_pdb_path"] = scaled_pdb
    state["tab4_last_view"] = v
    return v

def fetch_seq(_):
    try:
        with out_b2b:
            clear_output()
            if _need_acc(out_b2b):
                return
            print("Fetching sequence...")
        _pid, seq = fetch_sequence_aminoacids(state["acc"])
        state["sequence"] = seq
        with out_b2b:
            print(f"✅ Sequence length: {len(seq)} aa")
    except Exception as e:
        _err(out_b2b, e)

def run_b2b(_):
    try:
        if not state.get("sequence"):
            with out_b2b:
                clear_output()
                print("Fetch sequence first.")
            return

        btn_run_b2b.disabled = True
        with out_b2b:
            clear_output()
            print("Running Bio2Byte predictions (this can take a bit)...")

        pred = predict_biophysical_features(state["acc"], state["sequence"])
        state["bio2byte_raw"] = pred
        prot = pred.get("proteins", {}).get(state["acc"], {})
        dyn = pd.DataFrame(prot)

        # Only expose biological Bio2Byte features; ignore technical timing/runtime columns.
        num_cols = [c for c in dyn.columns if dyn[c].dtype.kind in "if" and "runtime" not in str(c).lower() and "execution_time" not in str(c).lower()]
        b2b_metric.options = num_cols
        if num_cols:
            b2b_metric.value = num_cols[0]

        state["dynamic_properties"] = dyn
        try:
            _sync_rin_b2b_metric_options()
        except Exception:
            pass
        try:
            _update_b2b_legend(); _update_rin_legend()
        except Exception:
            pass

        with out_b2b:
            print("✅ Done.")
            display(dyn.head())

    except Exception as e:
        _err(out_b2b, e)
    finally:
        btn_run_b2b.disabled = False

def show_b2b(_):
    with out_b2b:
        clear_output()
        if state.get("dynamic_properties") is None:
            print("Run predictions first.")
            return
        display_scrollable_df(state["dynamic_properties"], max_height="420px", max_width="95vw")

def show_b2b3d(_):
    try:
        with out_b2b:
            clear_output()

            if state.get("dynamic_properties") is None:
                print("Run predictions first.")
                return

            if not b2b_metric.value:
                print("Pick a metric.")
                return

            pdb_path = None
            chain = None
            uni_range = None
            pdb_to_uniprot = None

            if b2b_source.value == "af":
                if not state.get("af_path"):
                    print("Fetch AlphaFold model first (Tab 3).")
                    return
                pdb_path = state["af_path"]
                chain = "A"
                uni_range = None
                pdb_to_uniprot = None
            else:
                pdbid = (b2b_pdb_text.value or "").strip().upper() or (b2b_pdb_dropdown.value or "").strip().upper()
                if not pdbid:
                    print("Provide a PDB ID (type it or pick from the UniProt PDB dropdown).")
                    return
                pdb_path = download_rcsb_pdb(pdbid, out_dir=state["workdir"])

                chain = (b2b_chain_text.value or "").strip().upper() or None

                v = (b2b_chain_dropdown.value or "")
                if "|" in v:
                    parts = (v.split("|") + ["", ""])[:3]
                    _ch, u0, u1 = parts[0], parts[1], parts[2]
                    _ch = (_ch or "").strip().upper()[:1]
                    if _ch and not chain:
                        chain = _ch
                    if u0 and u1:
                        try:
                            uni_range = (int(u0), int(u1))
                        except Exception:
                            uni_range = None

                # Shared UniProt→PDB mapping, same logic used for PTM/variant rendering.
                pdb_to_uniprot, _uniprot_to_pdb = _build_structure_position_maps(
                    pdbid, chain=chain, uni_range=uni_range, pdb_path=pdb_path
                )
                if not pdb_to_uniprot:
                    with out_b2b:
                        print("⚠️ No residue-level UniProt↔PDB mapping found; using direct residue numbering fallback.")

            if viewer_mode.value == "ngl":
                v = display_b2b_3D_ngl(
                    state["dynamic_properties"],
                    pdb_path=pdb_path,
                    value_col=b2b_metric.value,
                    chain=chain,
                    uni_range=uni_range,
                    pdb_to_uniprot=pdb_to_uniprot,
                    site_overlay=b2b_site_overlay.value,
                    site_style=b2b_site_style.value
                )
                state["tab4_last_view"] = v
                state["tab4_last_source_pdb_path"] = pdb_path
                state["tab4_last_pdb_id"] = None if b2b_source.value == "af" else pdbid
                display(v)
                return

            # 4-panel py3Dmol
            view = display_b2b_4panel_py3dmol(
                state["dynamic_properties"],
                pdb_path=pdb_path,
                chain=chain,
                pdb_to_uniprot=pdb_to_uniprot
            )
            state["tab4_last_view"] = view
            state["tab4_last_source_pdb_path"] = pdb_path
            state["tab4_last_pdb_id"] = None if b2b_source.value == "af" else pdbid
            display(view)

    except Exception as e:
        _err(out_b2b, e)

import threading

# --- Auto-refresh controller for Tab 4 ---
state["b2b_autorefresh"] = True
_b2b_refresh_timer = None

def _b2b_autorefresh(_=None, delay=0.15):
    """Debounced auto refresh for Bio2Byte 3D panel."""
    global _b2b_refresh_timer
    if not state.get("b2b_autorefresh", True):
        return
    # Only auto-refresh if user already has predictions
    if state.get("dynamic_properties") is None:
        return

    # cancel pending
    try:
        if _b2b_refresh_timer is not None:
            _b2b_refresh_timer.cancel()
    except Exception:
        pass

    def _go():
        try:
            show_b2b3d(None)
        except Exception:
            pass

    _b2b_refresh_timer = threading.Timer(delay, _go)
    _b2b_refresh_timer.start()


btn_fetch_seq.on_click(fetch_seq)
btn_run_b2b.on_click(run_b2b)
btn_show_b2b.on_click(show_b2b)
btn_show_b2b3d.on_click(show_b2b3d)

def on_export_b2b(_):
    pdb_key = "tab4_last_colored_pdb_path" if state.get("tab4_last_colored_pdb_path") else "tab4_last_source_pdb_path"
    _export_current_view(
        f"tab4_bio2byte_{state.get('acc','protein')}_{b2b_metric.value or 'metric'}",
        "tab4_last_view",
        pdb_key,
        "tab4_last_pdb_id",
        out=out_b2b,
    )

btn_export_b2b.on_click(on_export_b2b)

# Auto-refresh whenever these change
viewer_mode.observe(_b2b_autorefresh, names="value")
b2b_metric.observe(_b2b_autorefresh, names="value")
b2b_metric.observe(_update_b2b_legend, names="value")
viewer_mode.observe(_update_b2b_legend, names="value")
b2b_site_overlay.observe(_b2b_autorefresh, names="value")
b2b_site_style.observe(_b2b_autorefresh, names="value")
b2b_source.observe(_b2b_autorefresh, names="value")

b2b_pdb_text.observe(_b2b_autorefresh, names="value")
b2b_chain_text.observe(_b2b_autorefresh, names="value")
b2b_pdb_dropdown.observe(_b2b_autorefresh, names="value")
b2b_chain_dropdown.observe(_b2b_autorefresh, names="value")


tab4_controls_pdb = w.VBox([
    w.HBox([b2b_pdb_text, b2b_chain_text]),
    w.HBox([b2b_pdb_dropdown, b2b_chain_dropdown]),
])
tab4_controls_pdb.layout.display = "none"

def _b2b_set_source_ui(*_):
    with out_b2b:
        clear_output()
        if b2b_source.value == "af":
            b2b_pdb_text.value = ""
            b2b_chain_text.value = ""
            try: b2b_pdb_dropdown.value = ""
            except Exception: pass
            try: b2b_chain_dropdown.value = ""
            except Exception: pass
            tab4_controls_pdb.layout.display = "none"
            print("AlphaFold selected. Use Tab 3 to fetch the AlphaFold model, then come back here to color by Bio2Byte properties.")
        else:
            tab4_controls_pdb.layout.display = ""
            print("PDB selected. Provide a PDB ID (or pick from UniProt PDB dropdown). Optional: choose chain/range from dropdown to restrict coloring.")

b2b_source.observe(lambda ch: _b2b_set_source_ui(), names="value")
_b2b_set_source_ui()

tab4 = w.VBox([
    w.HTML("<h3>Bio2Byte biophysical predictions</h3>"),
    w.HBox([btn_fetch_seq, btn_run_b2b, btn_show_b2b, btn_show_b2b3d, btn_export_b2b]),
    w.HBox([b2b_source, viewer_mode, b2b_metric]),
    w.HBox([b2b_site_overlay, b2b_site_style]),
    tab4_controls_pdb,
    b2b_legend,
    out_b2b
])

# Initialize Tab 4 dropdowns (safe)
try:
    _refresh_b2b_pdb_dropdowns()
except Exception:
    pass



# ----------------------------
# Shared UniProt PDB dropdown helpers for Tabs 5/6
# ----------------------------
def _pdb_options_from_uniprot_refs():
    refs = state.get("uniprot_pdb_refs") or []
    pdb_ids = sorted({(r.get("pdb_id") or "").upper() for r in refs if r.get("pdb_id")})
    return [("", "")] + [(pid, pid) for pid in pdb_ids]

def _chain_options_for_uniprot_pdb(pdb_id: str):
    pdb_id = (pdb_id or "").upper().strip()
    refs = state.get("uniprot_pdb_refs") or []
    cr = {}
    for r in refs:
        if (r.get("pdb_id") or "").upper() != pdb_id:
            continue
        for ch, rng in (r.get("chain_ranges") or {}).items():
            cr[ch] = rng
    opts = [("", "")]
    for ch in sorted(cr.keys()):
        rng = cr[ch]
        if rng and len(rng) == 2:
            opts.append((f"{ch} ({rng[0]}-{rng[1]})", ch))
        else:
            opts.append((f"{ch} (range n/a)", ch))
    return opts

def _selected_uniprot_chain_range(pdb_id: str, chain_id: str):
    pdb_id = (pdb_id or "").upper().strip()
    chain_id = (chain_id or "").upper().strip()[:1]
    if not pdb_id or not chain_id:
        return None
    for r in state.get("uniprot_pdb_refs") or []:
        if (r.get("pdb_id") or "").upper() != pdb_id:
            continue
        rng = (r.get("chain_ranges") or {}).get(chain_id)
        if rng and len(rng) == 2:
            return int(rng[0]), int(rng[1])
    return None

def _set_dd_options(dd, options):
    values = [v for _, v in options]
    old = dd.value
    dd.options = options
    dd.value = old if old in values else ""

def _sync_chain_dropdown_for_pdb(pdb_dd_widget, chain_dd_widget, range_label_widget=None):
    pid = (pdb_dd_widget.value or "").upper().strip()
    if not pid:
        chain_dd_widget.options = [("", "")]
        chain_dd_widget.value = ""
        if range_label_widget is not None:
            range_label_widget.value = ""
        return
    opts = _chain_options_for_uniprot_pdb(pid)
    _set_dd_options(chain_dd_widget, opts)
    if range_label_widget is not None:
        ch = (chain_dd_widget.value or "").upper().strip()[:1]
        rng = _selected_uniprot_chain_range(pid, ch)
        if ch and rng:
            range_label_widget.value = f"<b>UniProt range:</b> {ch} = {rng[0]}–{rng[1]}"
        elif ch:
            range_label_widget.value = f"<b>UniProt range:</b> {ch} = n/a"
        else:
            range_label_widget.value = ""

def _extract_chain_or_range(pdb_path: str, chain_id: str, out_path: str, start=None, end=None) -> str:
    chain_id = (chain_id or "A").strip()[:1]
    if start is None or end is None:
        start, end = chain_range_from_pdb(pdb_path, chain_id)
    parser = PDBParser(QUIET=True)
    io = PDBIO()
    io.set_structure(parser.get_structure("X", pdb_path))
    io.save(out_path, select=ChainRangeSelect(chain_id, int(start), int(end)))
    return out_path

# ----------------------------
# TAB 5: RIN (Residue Interaction Network)
# ----------------------------
out_rin = w.Output(layout={"border":"1px solid #ddd","padding":"6px"})
rin_legend = w.HTML("")  # dynamic RIN Bio2Byte colour legend
rin_source = w.RadioButtons(
    options=[("AlphaFold", "alphafold"), ("UniProt PDB dropdown", "uniprot_pdb"), ("Upload PDB", "upload")],
    value="alphafold",
    description="Structure:",
    layout=w.Layout(width="260px")
)
btn_dl_af = w.Button(description="Download AlphaFold PDB", button_style="warning")
rin_pdb_dd = w.Dropdown(options=[("", "")], value="", description="UniProt PDB:", layout=w.Layout(width="420px"))
rin_chain_dd = w.Dropdown(options=[("", "")], value="", description="Chain/range:", layout=w.Layout(width="420px"))
rin_range_lbl = w.HTML("")
upload_pdb = w.FileUpload(accept=".pdb", multiple=False, description="Upload PDB")
chain_rin = w.Text(description="Upload/AF chain:", placeholder="A", value="A", layout=w.Layout(width="220px"))
cutoff = w.FloatSlider(description="Cutoff Å:", min=4.0, max=12.0, step=0.5, value=8.0, readout=True)

rin_color_mode = w.Dropdown(
    description="Node color:",
    options=[("Site status (default)", "site"), ("Bio2Byte property + site border", "b2b")],
    value="site",
    layout=w.Layout(width="360px")
)
rin_b2b_metric = w.Dropdown(description="Bio2Byte:", options=[], layout=w.Layout(width="320px"))
rin_color_mode.observe(_update_rin_legend, names="value")
rin_b2b_metric.observe(_update_rin_legend, names="value")

def _sync_rin_b2b_metric_options():
    dyn = state.get("dynamic_properties")
    if dyn is None:
        rin_b2b_metric.options = []
        return
    cols = [c for c in dyn.columns if getattr(dyn[c], "dtype", None) is not None and dyn[c].dtype.kind in "if" and "runtime" not in str(c).lower() and "execution_time" not in str(c).lower()]
    rin_b2b_metric.options = cols
    if cols and rin_b2b_metric.value not in cols:
        rin_b2b_metric.value = cols[0]
btn_build_rin = w.Button(description="Build RIN", button_style="danger")
btn_show_rin = w.Button(description="Show RIN", button_style="success")

import os

def _save_upload_to_path(upl: w.FileUpload, out_dir: str) -> str:
    if not upl.value:
        raise ValueError("No PDB uploaded")
    v = upl.value
    if isinstance(v, dict):
        fname, item = next(iter(v.items()))
        name = item.get("metadata", {}).get("name") or item.get("name") or fname or "uploaded.pdb"
        content = item.get("content", b"")
    elif isinstance(v, (list, tuple)):
        item = v[0]
        name = item.get("metadata", {}).get("name") or item.get("name") or "uploaded.pdb"
        content = item.get("content", b"")
    else:
        raise TypeError(f"Unexpected FileUpload.value type: {type(v)}")
    if isinstance(content, memoryview):
        content = content.tobytes()
    elif isinstance(content, bytearray):
        content = bytes(content)
    os.makedirs(out_dir, exist_ok=True)
    fp = os.path.join(out_dir, name)
    with open(fp, "wb") as f:
        f.write(content)
    if os.path.getsize(fp) < 100:
        raise ValueError(f"Uploaded file looks too small ({os.path.getsize(fp)} bytes). Not a valid PDB?")
    return fp

def _refresh_rin_pdb_dropdowns():
    _set_dd_options(rin_pdb_dd, _pdb_options_from_uniprot_refs())
    _sync_chain_dropdown_for_pdb(rin_pdb_dd, rin_chain_dd, rin_range_lbl)

def _update_rin_chain_dropdown(*_):
    _sync_chain_dropdown_for_pdb(rin_pdb_dd, rin_chain_dd, rin_range_lbl)

def _update_rin_source_ui(*_):
    if rin_source.value == "uniprot_pdb":
        rin_uniprot_controls.layout.display = ""
        rin_upload_controls.layout.display = "none"
        btn_dl_af.layout.display = "none"
        chain_rin.description = "Fallback chain:"
    elif rin_source.value == "upload":
        rin_uniprot_controls.layout.display = "none"
        rin_upload_controls.layout.display = ""
        btn_dl_af.layout.display = "none"
        chain_rin.description = "Upload chain:"
    else:
        rin_uniprot_controls.layout.display = "none"
        rin_upload_controls.layout.display = "none"
        btn_dl_af.layout.display = ""
        chain_rin.description = "AF chain:"

rin_pdb_dd.observe(_update_rin_chain_dropdown, names="value")
rin_chain_dd.observe(_update_rin_chain_dropdown, names="value")
rin_source.observe(_update_rin_source_ui, names="value")

def dl_af(_):
    try:
        with out_rin:
            clear_output()
            if _need_acc(out_rin): 
                return
            print("Downloading AlphaFold PDB...")
        fp = download_alphafold_pdb(state["acc"], state["workdir"])
        state["rin_pdb_path"] = fp
        with out_rin:
            print(f"✅ Saved: {fp}")
    except Exception as e:
        _err(out_rin, e)

def _resolve_rin_structure():
    src = rin_source.value
    if src == "upload":
        pdb_path = _save_upload_to_path(upload_pdb, state["workdir"])
        ch = (chain_rin.value or "A").strip()[:1]
        rng = None
        label = f"uploaded {os.path.basename(pdb_path)}"
    elif src == "uniprot_pdb":
        pid = (rin_pdb_dd.value or "").upper().strip()
        if not pid:
            raise ValueError("Select a UniProt PDB entry first.")
        pdb_path = _download_pdb_to_workdir(pid, state["workdir"])
        ch = (rin_chain_dd.value or chain_rin.value or "A").strip()[:1]
        rng = _selected_uniprot_chain_range(pid, ch)
        label = f"{pid} chain {ch}"
    else:
        if state.get("rin_pdb_path") and os.path.exists(state["rin_pdb_path"]):
            pdb_path = state["rin_pdb_path"]
        else:
            pdb_path = download_alphafold_pdb(state["acc"], state["workdir"])
            state["rin_pdb_path"] = pdb_path
        ch = (chain_rin.value or "A").strip()[:1]
        rng = None
        label = f"AlphaFold chain {ch}"
    return pdb_path, ch, rng, label

def nx_rin_to_pyvis_b2b_overlay(G, dyn_df, value_col, ptm_positions=None, mutation_positions=None, out_html="rin_pyvis_b2b.html", highlight_resns=None, highlight_resi=None):
    """PyVis RIN: node fill = Bio2Byte value, border/size = PTM/variant status."""
    vals = pd.to_numeric(dyn_df[value_col], errors="coerce")
    vmin, vmax = float(vals.min()), float(vals.max())
    ptm_set = set(int(x) for x in (ptm_positions or []))
    mut_set = set(int(x) for x in (mutation_positions or []))
    hl_resns = set((highlight_resns or ()))
    hl_resi = set(int(x) for x in (highlight_resi or []))

    net = Network(height="600px", width="100%", directed=False, notebook=True, cdn_resources="in_line", select_menu=True, filter_menu=False)
    net.set_options(r'''
    {"physics":{"stabilization":true},"interaction":{"hover":true},"nodes":{"borderWidth":2}}
    ''')

    for (ch, resi), attrs in G.nodes(data=True):
        resi = int(resi)
        resn = attrs.get("ResName", "")
        idx = resi - 1
        if 0 <= idx < len(vals) and pd.notna(vals.iloc[idx]):
            value = float(vals.iloc[idx])
            bg = "#" + pseudocolor(vmin, vmax, value).replace("0x", "")
            val_txt = f"{value:.3f}"
            interp = _b2b_interpretation(value_col, value)
        else:
            bg = "#B0B0B0"
            val_txt = "n/a"
            interp = None

        is_ptm = resi in ptm_set
        is_mut = resi in mut_set
        if is_ptm and is_mut:
            border, bw, size, status = "#9467bd", 3, 42, "PTM+Variant"
        elif is_ptm:
            border, bw, size, status = "#d62728", 3, 36, "PTM"
        elif is_mut:
            border, bw, size, status = "#1f77b4", 3, 36, "Variant"
        else:
            border, bw, size, status = "#000000", 1, 30, "Other"   # thin black border for 'other'

        # residue-selection highlight -> dashed border
        highlighted = (str(resn).upper() in hl_resns) or (resi in hl_resi)
        node_kwargs = {"borderWidth": bw}
        title_extra = ""
        if highlighted:
            if bw == 0:
                border = "#000000"
            node_kwargs["borderWidth"] = max(bw, 3)
            node_kwargs["shapeProperties"] = {"borderDashes": [6, 4]}
            title_extra = " | selected"

        metric_lbl = interp if interp else value_col
        node_id = f"{ch}:{resi}"
        net.add_node(
            node_id,
            label=f"{resn} {resi}" if resn else str(resi),
            title=f"{resn} {ch}{resi} | {status} | {metric_lbl}: {val_txt}{title_extra}",
            color={"background": bg, "border": border, "highlight": {"background": bg, "border": border}, "hover": {"background": bg, "border": border}},
            size=size,
            **node_kwargs,
        )

    for u, v, attrs in G.edges(data=True):
        net.add_edge(f"{u[0]}:{int(u[1])}", f"{v[0]}:{int(v[1])}", value=float(attrs.get("weight", 1.0)), title=str(attrs.get("contact", "")))

    net.write_html(out_html, notebook=False, open_browser=False)
    return out_html


def _current_rin_site_positions():
    ptm_pos = None
    mut_pos = None
    if state.get("ptm_table") is not None and "position" in state["ptm_table"].columns:
        ptm_pos = list(set(state["ptm_table"]["position"].dropna().astype(int).tolist()))
    if state.get("variants_df") is not None and "position" in state["variants_df"].columns:
        mut_pos = list(set(state["variants_df"]["position"].dropna().astype(int).tolist()))
    return ptm_pos, mut_pos

def _parse_int_positions(text):
    """Parse '905, 910-920' into a set of ints."""
    out = set()
    for part in re.split(r"[,\s]+", (text or "").strip()):
        if not part:
            continue
        if "-" in part:
            try:
                a, b = part.split("-", 1)
                a, b = int(a), int(b)
                out.update(range(min(a, b), max(a, b) + 1))
            except Exception:
                pass
        else:
            try:
                out.add(int(part))
            except Exception:
                pass
    return out

def _render_cached_rin_html():
    G = state.get("rin_graph")
    if G is None:
        raise ValueError("No cached RIN graph yet. Build the RIN once first.")
    ptm_pos, mut_pos = _current_rin_site_positions()
    _sync_rin_b2b_metric_options()

    # residue-selection highlight sets (by AA type and by UniProt position)
    hl_resns = set(rin_highlight_aa.value or ())
    hl_uni = _parse_int_positions(rin_highlight_pos.value)
    u2p = state.get("rin_uniprot_to_pdb")
    if hl_uni and u2p:
        hl_resi = set()
        for p in hl_uni:
            hl_resi.update(int(x) for x in u2p.get(int(p), []))
    else:
        hl_resi = set(int(p) for p in hl_uni)  # AlphaFold/upload: node resi == UniProt

    suffix = (rin_b2b_metric.value or "site").replace("/", "_").replace(" ", "_") if rin_color_mode.value == "b2b" else "site"
    out_html = os.path.join(state["workdir"], f"rin_pyvis_{state['acc'] or 'session'}_{rin_color_mode.value}_{suffix}.html")
    if rin_color_mode.value == "b2b" and state.get("dynamic_properties") is not None and rin_b2b_metric.value:
        html_path = nx_rin_to_pyvis_b2b_overlay(
            G,
            state["dynamic_properties"],
            rin_b2b_metric.value,
            ptm_positions=ptm_pos,
            mutation_positions=mut_pos,
            out_html=out_html,
            highlight_resns=hl_resns,
            highlight_resi=hl_resi,
        )
    else:
        html_path = nx_rin_to_pyvis_default(G, ptm_positions=ptm_pos, mutation_positions=mut_pos, out_html=out_html,
                                            highlight_resns=hl_resns, highlight_resi=hl_resi)
    state["rin_html"] = html_path
    return html_path
def build_rin(_):
    try:
        with out_rin:
            clear_output()
            if _need_acc(out_rin):
                return
            pdb_path, ch, rng, label = _resolve_rin_structure()
            print("Using pdb_path:", pdb_path)
            print("File size (bytes):", os.path.getsize(pdb_path))
            if rng:
                print(f"Building RIN from {label}, UniProt mapped range {rng[0]}-{rng[1]}, cutoff {cutoff.value} Å ...")
                rin_input = os.path.join(state["workdir"], f"rin_{os.path.basename(pdb_path).replace('.pdb','')}_{ch}_{rng[0]}_{rng[1]}.pdb")
                _extract_chain_or_range(pdb_path, ch, rin_input, rng[0], rng[1])
            else:
                print(f"Building RIN from {label}, cutoff {cutoff.value} Å ...")
                rin_input = pdb_path

        tmp = build_geometry_graph_from_pdb(rin_input, chain=ch, cutoff=float(cutoff.value))
        G = tmp[0] if isinstance(tmp, tuple) else tmp
        state["rin_graph"] = G
        state["rin_graph_meta"] = {"pdb_path": pdb_path, "chain": ch, "range": rng, "label": label, "cutoff": float(cutoff.value)}
        # residue mapping for position-based highlight (UniProt -> node/PDB resi)
        try:
            if rin_source.value == "uniprot_pdb":
                _pid = (rin_pdb_dd.value or "").upper().strip()
                _p2u, _u2p = _build_structure_position_maps(_pid, chain=ch, uni_range=None, pdb_path=pdb_path)
                state["rin_uniprot_to_pdb"] = _u2p or None
            else:
                state["rin_uniprot_to_pdb"] = None
        except Exception:
            state["rin_uniprot_to_pdb"] = None
        print("Chain used for RIN:", G.graph.get("chain"))
        _render_cached_rin_html()

        with out_rin:
            print("✅ RIN built and cached.")
            print("Change the node-color/property dropdowns and click 'Show RIN' to recolor without rebuilding.")

    except Exception as e:
        _err(out_rin, e)

from IPython.display import HTML

def show_rin(_):
    with out_rin:
        clear_output()
        try:
            # Re-render from the cached graph so changing Bio2Byte property/node colour does not require rebuilding contacts.
            if state.get("rin_graph") is not None:
                p = _render_cached_rin_html()
            else:
                p = state.get("rin_html")
            if not p or not os.path.exists(p):
                print("No RIN HTML yet. Build it first.")
                return
            with open(p, "r", encoding="utf-8") as f:
                html = f.read()
            display(HTML(html))
        except Exception as e:
            print("ERROR:", e)

btn_dl_af.on_click(dl_af)
btn_build_rin.on_click(build_rin)
btn_show_rin.on_click(show_rin)

rin_uniprot_controls = w.VBox([w.HBox([rin_pdb_dd, rin_chain_dd]), rin_range_lbl])
rin_upload_controls = w.HBox([upload_pdb])
_update_rin_source_ui()
try:
    _refresh_rin_pdb_dropdowns()
except Exception:
    pass

btn_export_rin = w.Button(description="Export RIN HTML", button_style="", tooltip="Export the current/cached RIN as HTML")

def on_export_rin(_):
    with out_rin:
        p = state.get("rin_html")
        if state.get("rin_graph") is not None:
            try:
                p = _render_cached_rin_html()
            except Exception:
                pass
        if p and os.path.exists(p):
            dst = os.path.join(state.get("workdir", tempfile.gettempdir()), f"tab5_rin_{_safe_token(state.get('acc','protein'))}.html")
            try:
                shutil.copyfile(p, dst)
                print("✅ Exported RIN:")
                _display_direct_download(dst, "RIN HTML")
            except Exception as e:
                print("⚠️ RIN export failed:", e)
        else:
            print("No RIN HTML yet. Build/show the RIN first.")

btn_export_rin.on_click(on_export_rin)

rin_highlight_aa = w.SelectMultiple(
    options=[("Ser","SER"),("Thr","THR"),("Tyr","TYR"),("Cys","CYS"),("Lys","LYS"),
             ("His","HIS"),("Asp","ASP"),("Glu","GLU"),("Arg","ARG"),("Asn","ASN"),("Gln","GLN")],
    value=(), description="Highlight AA:", rows=4, layout=w.Layout(width="230px"))
rin_highlight_pos = w.Text(value="", description="Highlight pos:",
                           placeholder="e.g. 905, 910-920 (UniProt)", layout=w.Layout(width="380px"))
rin_highlight_note = w.HTML("<span style='font-size:11px;color:#666;'>Selected residues get a dashed border. "
                            "Positions are UniProt numbering (mapped to PDB automatically). Click 'Show RIN' to apply.</span>")

tab5 = w.VBox([
    w.HTML("<h3>Residue Interaction Network (PyVis)</h3>"),
    w.HBox([rin_source, btn_dl_af]),
    rin_uniprot_controls,
    rin_upload_controls,
    w.HBox([chain_rin, cutoff]),
    w.HBox([rin_color_mode, rin_b2b_metric]),
    w.HBox([rin_highlight_aa, rin_highlight_pos]),
    rin_highlight_note,
    w.HBox([btn_build_rin, btn_show_rin, btn_export_rin]),
    rin_legend,
    out_rin
])

# ----------------------------
# TAB 6: TM-align tool with upload or UniProt PDB dropdowns
# ----------------------------
tm_source1 = w.RadioButtons(
    options=[("AlphaFold", "alphafold"), ("UniProt PDB dropdown", "uniprot_pdb"), ("Upload PDB", "upload")],
    value="uniprot_pdb",
    description="Structure 1:",
    layout=w.Layout(width="260px")
)
tm_source2 = w.RadioButtons(
    options=[("AlphaFold", "alphafold"), ("UniProt PDB dropdown", "uniprot_pdb"), ("Upload PDB", "upload")],
    value="uniprot_pdb",
    description="Structure 2:",
    layout=w.Layout(width="260px")
)
tm_pdb1_dd = w.Dropdown(options=[("", "")], value="", description="PDB 1:", layout=w.Layout(width="390px"))
tm_pdb2_dd = w.Dropdown(options=[("", "")], value="", description="PDB 2:", layout=w.Layout(width="390px"))
tm_chain1_dd = w.Dropdown(options=[("", "")], value="", description="Chain 1:", layout=w.Layout(width="390px"))
tm_chain2_dd = w.Dropdown(options=[("", "")], value="", description="Chain 2:", layout=w.Layout(width="390px"))
tm_range1_lbl = w.HTML("")
tm_range2_lbl = w.HTML("")

tm_site_overlay = w.Dropdown(
    description="Highlight sites:",
    options=[("No site overlay", "none"), ("PTMs", "ptm"), ("Variants", "mut"), ("PTM + variant overlap", "overlap"), ("All PTM/variant sites", "both")],
    value="both",
    layout=w.Layout(width="340px")
)

tm_site_style = w.Dropdown(
    description="Site style:",
    options=[("Sticks/licorice", "stick"), ("Sphere", "sphere"), ("Ball+stick", "ballstick")],
    value="stick",
    layout=w.Layout(width="300px")
)

tm_site_target = w.Dropdown(
    description="Sites on:",
    options=[("Both structures", "both"), ("Only Structure 1", "s1"), ("Only Structure 2", "s2")],
    value="both",
    layout=w.Layout(width="340px")
)

tm_view_region = w.Dropdown(
    description="Show region:",
    options=[("Show all (full structures)", "all"), ("Aligned region only", "aligned")],
    value="all",
    layout=w.Layout(width="320px")
)

# Reuse the original upload/range widgets, but relabel the upload controls.
upload1.description = "Upload PDB 1"
upload2.description = "Upload PDB 2"
chain1.description = "Chain 1"
chain2.description = "Chain 2"

out_tm = out
btn_export_tm = w.Button(description="Export alignment view/PDBs", button_style="", tooltip="Export last TM-align view and aligned/reference PDB files")
workdir = state.get("workdir") or workdir

def _refresh_tm_pdb_dropdowns():
    opts = _pdb_options_from_uniprot_refs()
    _set_dd_options(tm_pdb1_dd, opts)
    _set_dd_options(tm_pdb2_dd, opts)
    _sync_chain_dropdown_for_pdb(tm_pdb1_dd, tm_chain1_dd, tm_range1_lbl)
    _sync_chain_dropdown_for_pdb(tm_pdb2_dd, tm_chain2_dd, tm_range2_lbl)

def _update_tm_chain1(*_):
    _sync_chain_dropdown_for_pdb(tm_pdb1_dd, tm_chain1_dd, tm_range1_lbl)
    if tm_chain1_dd.value:
        chain1.value = tm_chain1_dd.value
        rng = _selected_uniprot_chain_range(tm_pdb1_dd.value, tm_chain1_dd.value)
        if rng:
            start1.value, end1.value = rng[0], rng[1]

def _update_tm_chain2(*_):
    _sync_chain_dropdown_for_pdb(tm_pdb2_dd, tm_chain2_dd, tm_range2_lbl)
    if tm_chain2_dd.value:
        chain2.value = tm_chain2_dd.value
        rng = _selected_uniprot_chain_range(tm_pdb2_dd.value, tm_chain2_dd.value)
        if rng:
            start2.value, end2.value = rng[0], rng[1]

def _update_tm_source_ui(*_):
    tm1_uniprot_controls.layout.display = "" if tm_source1.value == "uniprot_pdb" else "none"
    tm1_upload_controls.layout.display = "" if tm_source1.value == "upload" else "none"
    tm2_uniprot_controls.layout.display = "" if tm_source2.value == "uniprot_pdb" else "none"
    tm2_upload_controls.layout.display = "" if tm_source2.value == "upload" else "none"

    # AlphaFold uses chain A by default and the full model range is auto-filled at run time.
    if tm_source1.value == "alphafold":
        chain1.value = "A"
    if tm_source2.value == "alphafold":
        chain2.value = "A"

tm_pdb1_dd.observe(_update_tm_chain1, names="value")
tm_chain1_dd.observe(_update_tm_chain1, names="value")
tm_pdb2_dd.observe(_update_tm_chain2, names="value")
tm_chain2_dd.observe(_update_tm_chain2, names="value")
tm_source1.observe(_update_tm_source_ui, names="value")
tm_source2.observe(_update_tm_source_ui, names="value")

def _resolve_tm_structure(which: int):
    source = tm_source1.value if which == 1 else tm_source2.value

    if source == "alphafold":
        if not state.get("acc"):
            raise ValueError("Set a UniProt accession first before using AlphaFold in TM-align.")

        # Reuse the AlphaFold model fetched in Tab 3 when available.
        # If it has not been fetched yet, download it here so Tab 6 works independently.
        pdb = state.get("af_path")
        if not pdb or not os.path.exists(pdb):
            pdb = download_alphafold_pdb(state["acc"], out_dir=state["workdir"])
            state["af_path"] = pdb

        ch = (chain1.value if which == 1 else chain2.value) or "A"
        ch = ch.strip()[:1] or "A"
        s0, e0 = chain_range_from_pdb(pdb, ch)

        if which == 1:
            chain1.value, start1.value, end1.value = ch, int(s0), int(e0)
        else:
            chain2.value, start2.value, end2.value = ch, int(s0), int(e0)

        label = f"AlphaFold_{state['acc']}_{ch}"
        return pdb, ch, int(s0), int(e0), label

    if which == 1:
        if source == "upload":
            pdb = save_upload(upload1, workdir)
            ch = (chain1.value or "A").strip()[:1]
            s0, e0 = int(start1.value), int(end1.value)
            label = os.path.basename(pdb)
        else:
            pid = (tm_pdb1_dd.value or "").upper().strip()
            if not pid:
                raise ValueError("Select PDB 1 from the UniProt dropdown, switch Structure 1 to AlphaFold, or upload a PDB.")
            pdb = _download_pdb_to_workdir(pid, workdir)
            ch = (tm_chain1_dd.value or chain1.value or "A").strip()[:1]
            rng = _selected_uniprot_chain_range(pid, ch)
            s0, e0 = rng if rng else chain_range_from_pdb(pdb, ch)
            chain1.value, start1.value, end1.value = ch, int(s0), int(e0)
            label = f"{pid}_{ch}"
    else:
        if source == "upload":
            pdb = save_upload(upload2, workdir)
            ch = (chain2.value or "A").strip()[:1]
            s0, e0 = int(start2.value), int(end2.value)
            label = os.path.basename(pdb)
        else:
            pid = (tm_pdb2_dd.value or "").upper().strip()
            if not pid:
                raise ValueError("Select PDB 2 from the UniProt dropdown, switch Structure 2 to AlphaFold, or upload a PDB.")
            pdb = _download_pdb_to_workdir(pid, workdir)
            ch = (tm_chain2_dd.value or chain2.value or "A").strip()[:1]
            rng = _selected_uniprot_chain_range(pid, ch)
            s0, e0 = rng if rng else chain_range_from_pdb(pdb, ch)
            chain2.value, start2.value, end2.value = ch, int(s0), int(e0)
            label = f"{pid}_{ch}"
    return pdb, ch, int(s0), int(e0), label

def autofill_ranges(_):
    out.clear_output()
    with out:
        try:
            pdb1, ch1, s1, e1, label1 = _resolve_tm_structure(1)
            pdb2, ch2, s2, e2, label2 = _resolve_tm_structure(2)
            start1.value, end1.value = s1, e1
            start2.value, end2.value = s2, e2
            print(f"Ranges auto-filled: {label1} {ch1}:{s1}-{e1}; {label2} {ch2}:{s2}-{e2}")
        except Exception as e:
            print("ERROR:", e)

btn_range.on_click(autofill_ranges)

def _aligned_region_selection():
    """NGL selection for the aligned region = overlap of the two input ranges.

    Returns e.g. "700-1012", or "protein" if the overlap is unknown/empty. Both
    structures share UniProt numbering, so the same range applies to each.
    """
    rng = state.get("tab6_aligned_range")
    if not rng:
        return "protein"
    lo, hi = rng
    return f"{int(lo)}-{int(hi)}"


def visualize_tmalign_ngl_sites(pdb_ref, pdb_aligned, label_ref="Structure 2/reference", label_aligned="Structure 1/aligned", site_overlay="both", site_style="stick", chain_ref=None, chain_aligned=None, site_target="both", view_region="all"):
    """Render TM-align as a fresh standalone NGL HTML iframe every time.

    pdb_ref     = structure 2 (reference)            -> component 0, blue
    pdb_aligned = structure 1 transformed into 2's frame -> component 1, red
    site_target controls which structure(s) get grey site markers: both / s1 / s2.
    view_region "all" shows full structures; "aligned" shows only the structurally
                superposed residues (CA proximity computed at align time).
    """
    run_id = uuid.uuid4().hex[:8]
    html_path = os.path.join(state.get("workdir", tempfile.gettempdir()), f"tab6_tmalign_live_{run_id}.html")

    if (view_region or "all").lower() == "aligned":
        sel_ref_base = _aligned_region_selection()
        sel_aln_base = _aligned_region_selection()
    else:
        sel_ref_base = "protein"
        sel_aln_base = "protein"

    reps_ref = [{"type": "cartoon", "params": {"sele": sel_ref_base, "color": "#1f77b4", "opacity": 0.6}}]
    reps_aln = [{"type": "cartoon", "params": {"sele": sel_aln_base, "color": "#d62728", "opacity": 0.85}}]

    overlay_mode = (site_overlay or "none").lower()
    if overlay_mode != "none":
        sites = _mapped_site_residues_for_export(overlay_mode, pdb_to_uniprot=None)
        # "Aligned region only": keep just the sites inside the overlap range [lo, hi].
        # "Show all": keep every site. Both structures share UniProt numbering, so
        # the same residue numbers apply to each. We filter here (not via an NGL
        # "and" expression) and emit a plain residue selection that renders reliably.
        if (view_region or "all").lower() == "aligned":
            rng = state.get("tab6_aligned_range")
            if rng:
                lo, hi = rng
                sites = [s for s in sites if lo <= int(s) <= hi]
        rep = _site_representation_name(site_style)
        site_sel = _ngl_selection_from_residues(sites, chain=None)
        if site_sel:
            target = (site_target or "both").lower()
            # component 1 (red) = Structure 1; component 0 (blue) = Structure 2
            if target in ("both", "s1"):
                reps_aln.append({"type": rep, "params": {"sele": site_sel, "color": "#7a7a7a"}})
            if target in ("both", "s2"):
                reps_ref.append({"type": rep, "params": {"sele": site_sel, "color": "#7a7a7a"}})

    note = (
        f"TM-align view | blue = {html.escape(str(label_ref))}; red = {html.escape(str(label_aligned))}; "
        f"sites on = {html.escape(str(site_target))}; region = {html.escape(str(view_region))}; style = {html.escape(str(site_style))}"
    )
    _write_standalone_ngl_html(
        html_path,
        [pdb_ref, pdb_aligned],
        title="Scop3P TM-align view",
        components=[{"representations": reps_ref}, {"representations": reps_aln}],
        note=note,
    )
    state["tab6_last_live_html"] = html_path
    return _html_iframe_from_file(html_path, width="100%", height=650)


def _display_last_tmalign_view(clear=True):
    """Redraw the last TM-align result with the currently selected highlight settings."""
    ref = state.get("tab6_last_ref_pdb_path")
    aln = state.get("tab6_last_aligned_pdb_path")
    if not (ref and aln and os.path.exists(ref) and os.path.exists(aln)):
        return False
    if clear:
        out.clear_output()
    with out:
        summary = state.get("tab6_last_summary")
        if summary:
            print(summary)
        label_ref = state.get("tab6_last_label_ref", "Structure 2/reference")
        label_aln = state.get("tab6_last_label_aligned", "Structure 1/aligned")
        print(f"Color legend: blue = {label_ref} (Structure 2); red = {label_aln} (Structure 1); grey = selected sites.")
        overlay_label = dict(tm_site_overlay.options).get(tm_site_overlay.value, tm_site_overlay.value)
        style_label = dict(tm_site_style.options).get(tm_site_style.value, tm_site_style.value)
        target_label = dict(tm_site_target.options).get(tm_site_target.value, tm_site_target.value)
        region_label = dict(tm_view_region.options).get(tm_view_region.value, tm_view_region.value)
        region_line = f"Sites: {overlay_label} | on: {target_label} | style: {style_label} | region: {region_label}"
        rng = state.get("tab6_aligned_range")
        if (tm_view_region.value or "all").lower() == "aligned" and rng:
            region_line += f" ({rng[0]}-{rng[1]})"
        print(region_line)
        tm_view = visualize_tmalign_ngl_sites(
            ref,
            aln,
            label_ref=label_ref,
            label_aligned=label_aln,
            site_overlay=tm_site_overlay.value,
            site_style=tm_site_style.value,
            chain_ref=state.get("tab6_last_chain_ref"),
            chain_aligned=state.get("tab6_last_chain_aligned"),
            site_target=tm_site_target.value,
            view_region=tm_view_region.value,
        )
        state["tab6_last_view"] = tm_view
        display(tm_view)
    return True


def run_align(_):
    out.clear_output()
    with out:
        try:
            pdb1, ch1, s1, e1, label1 = _resolve_tm_structure(1)
            pdb2, ch2, s2, e2, label2 = _resolve_tm_structure(2)

            run_id = uuid.uuid4().hex[:8]
            seg1 = os.path.join(workdir, f"seg1_{run_id}.pdb")
            seg2 = os.path.join(workdir, f"seg2_{run_id}.pdb")

            _extract_chain_or_range(pdb1, ch1, seg1, s1, e1)
            _extract_chain_or_range(pdb2, ch2, seg2, s2, e2)

            # Get TM-align's rotation matrix (Chain_1 -> Chain_2) and apply it to seg1
            # ourselves. The resulting file contains ONLY structure 1, superposed into
            # structure 2's frame, so we never depend on chain IDs inside the -o file.
            matrix_path, stdout = run_tmalign_matrix(seg1, seg2, out_dir=workdir, tag=run_id)
            aligned_only = os.path.join(workdir, f"aligned_only_{run_id}.pdb")
            _transform_pdb_with_tmalign_matrix(seg1, aligned_only, matrix_path)

            # Aligned region = overlap of the two input residue ranges
            # (both in shared UniProt numbering): [max(start1,start2), min(end1,end2)].
            try:
                lo = max(int(s1), int(s2))
                hi = min(int(e1), int(e2))
                state["tab6_aligned_range"] = (lo, hi) if lo <= hi else None
            except Exception:
                state["tab6_aligned_range"] = None

            # Parse RMSD / TM-score / aligned length from the TM-align output.
            m_ali = re.search(r"Aligned length=\s*(\d+)", stdout)
            m_rmsd = re.search(r"RMSD=\s*([\d.]+)", stdout)
            tms = re.findall(r"TM-score=\s*([\d.]+)", stdout)
            metrics = []
            if m_ali:
                metrics.append(f"Aligned length = {m_ali.group(1)}")
            if m_rmsd:
                metrics.append(f"RMSD = {m_rmsd.group(1)} \u00c5")
            if len(tms) >= 2:
                metrics.append(f"TM-score = {tms[0]} (norm. {label1}), {tms[1]} (norm. {label2})")
            elif tms:
                metrics.append(f"TM-score = {tms[0]}")
            metric_line = " | ".join(metrics) if metrics else next((ln for ln in stdout.splitlines() if ln.strip()), "TM-align finished.")
            state["tab6_last_ref_pdb_path"] = seg2
            state["tab6_last_aligned_pdb_path"] = aligned_only
            state["tab6_last_label_ref"] = label2
            state["tab6_last_label_aligned"] = label1
            state["tab6_last_chain_ref"] = ch2
            state["tab6_last_chain_aligned"] = ch1
            state["tab6_last_summary"] = f"Aligned {label1} chain {ch1}:{s1}-{e1} against {label2} chain {ch2}:{s2}-{e2}\n{metric_line}"

            # Label the "Sites on" dropdown with the actual structure identities.
            tm_site_target.options = [
                ("Both structures", "both"),
                (f"Only {label1} (Structure 1)", "s1"),
                (f"Only {label2} (Structure 2)", "s2"),
            ]

        except Exception as e:
            print("ERROR:", e)
            return

    _display_last_tmalign_view(clear=True)

btn_run.on_click(run_align)

def _refresh_tmalign_overlay_on_change(*_):
    # When the user changes Highlight sites or Site style after an alignment,
    # rebuild only the NGL view from cached aligned/reference PDBs.
    _display_last_tmalign_view(clear=True)

tm_site_overlay.observe(_refresh_tmalign_overlay_on_change, names="value")
tm_site_style.observe(_refresh_tmalign_overlay_on_change, names="value")
tm_site_target.observe(_refresh_tmalign_overlay_on_change, names="value")
tm_view_region.observe(_refresh_tmalign_overlay_on_change, names="value")

def on_export_tm(_):
    with out:
        live_html = state.get("tab6_last_live_html")
        prefix = f"tab6_tmalign_{_safe_token(state.get('acc','protein'))}"
        if live_html and os.path.exists(live_html):
            dst_html = os.path.join(state.get("workdir", tempfile.gettempdir()), f"{prefix}.html")
            try:
                shutil.copyfile(live_html, dst_html)
                print("✅ Exported TM-align view:")
                _display_direct_download(dst_html, "TM-align HTML view")
            except Exception as e:
                print("⚠️ TM-align HTML export failed:", e)
        else:
            print("No TM-align HTML view yet. Run Align + visualize first.")
        aln = state.get("tab6_last_aligned_pdb_path")
        if aln and os.path.exists(aln):
            dst = os.path.join(state.get("workdir", tempfile.gettempdir()), f"{prefix}_aligned.pdb")
            try:
                shutil.copyfile(aln, dst)
                _display_direct_download(dst, "Aligned PDB")
            except Exception as e:
                print("⚠️ Aligned PDB export failed:", e)
        ref = state.get("tab6_last_ref_pdb_path")
        if ref and os.path.exists(ref):
            dst = os.path.join(state.get("workdir", tempfile.gettempdir()), f"{prefix}_reference.pdb")
            try:
                shutil.copyfile(ref, dst)
                _display_direct_download(dst, "Reference PDB")
            except Exception as e:
                print("⚠️ Reference PDB export failed:", e)

btn_export_tm.on_click(on_export_tm)

tm1_uniprot_controls = w.VBox([w.HBox([tm_pdb1_dd, tm_chain1_dd]), tm_range1_lbl])
tm2_uniprot_controls = w.VBox([w.HBox([tm_pdb2_dd, tm_chain2_dd]), tm_range2_lbl])
tm1_upload_controls = w.HBox([upload1])
tm2_upload_controls = w.HBox([upload2])
_update_tm_source_ui()
try:
    _refresh_tm_pdb_dropdowns()
except Exception:
    pass

_tm_struct1_box = w.VBox([
    w.HTML("<div style='background:#fdecea;border-left:6px solid #d62728;padding:5px 10px;font-weight:bold;color:#5c1313;'>Structure 1 \u2014 shown in red</div>"),
    tm_source1,
    tm1_uniprot_controls,
    tm1_upload_controls,
    w.HBox([chain1, start1, end1]),
], layout=w.Layout(border="1px solid #f3b0ab", padding="6px", margin="0 0 8px 0"))

_tm_struct2_box = w.VBox([
    w.HTML("<div style='background:#e8f0fe;border-left:6px solid #1f77b4;padding:5px 10px;font-weight:bold;color:#13315c;'>Structure 2 \u2014 shown in blue</div>"),
    tm_source2,
    tm2_uniprot_controls,
    tm2_upload_controls,
    w.HBox([chain2, start2, end2]),
], layout=w.Layout(border="1px solid #9ec3ff", padding="6px", margin="0 0 8px 0"))

tmalign_ui = widgets.VBox([
    _tm_struct1_box,
    _tm_struct2_box,
    widgets.HBox([tm_site_overlay, tm_site_style]),
    widgets.HBox([tm_site_target, tm_view_region]),
    widgets.HBox([btn_range, btn_run, btn_export_tm]),
    out,
])

tab6 = w.VBox([
    w.HTML("<h3>TM-align (AlphaFold, UniProt PDB dropdown, or uploaded structures)</h3>"),
    w.HTML("<p><b>Note:</b> TM-align must be installed in the container/server and available on PATH as <code>TM-align</code>.</p>"),
    tmalign_ui,
])

# ----------------------------
# Assemble tabs
# ----------------------------
# ---- Data availability: probe, badges, and structure-control gating ----
def _http_head_ok(url, timeout=6):
    try:
        import requests
        r = requests.head(url, timeout=timeout, allow_redirects=True)
        if r.status_code == 200:
            return True
        r = requests.get(url, timeout=timeout, stream=True)
        ok = r.status_code == 200
        r.close()
        return ok
    except Exception:
        return None  # unknown (network error)

def _alphafold_available(acc):
    return _http_head_ok(f"https://alphafold.ebi.ac.uk/files/AF-{acc}-F1-model_v6.pdb")

def _set_enabled(widget_list, enabled):
    for _wdg in widget_list:
        try:
            _wdg.disabled = not enabled
        except Exception:
            pass

def _avail_badge(label, ok, detail=""):
    if ok is None:
        bg, fg, mark = "#eeeeee", "#555555", "?"
    elif ok:
        bg, fg, mark = "#e6f4ea", "#137333", "\u2713"
    else:
        bg, fg, mark = "#fce8e6", "#a50e0e", "\u2717"
    txt = f"{mark} {label}" + (f": {detail}" if detail else "")
    return (f"<span style='display:inline-block;margin:2px 6px 2px 0;padding:2px 9px;"
            f"border-radius:11px;background:{bg};color:{fg};font-size:12px;font-weight:600;'>{txt}</span>")

def render_availability_strip():
    av = state.get("avail", {})
    ptm = av.get("ptm"); var = av.get("var"); pdb = av.get("pdb")
    parts = [
        _avail_badge("AlphaFold", av.get("af")),
        _avail_badge("PDB structures", (pdb or 0) > 0 if pdb is not None else None, "0" if not pdb else str(pdb)),
        _avail_badge("PTMs", None if ptm is None else ptm > 0, "not fetched" if ptm is None else str(ptm)),
        _avail_badge("Variants", None if var is None else var > 0, "not fetched" if var is None else str(var)),
    ]
    availability_box.value = ("<div style='margin:6px 0;padding:4px 2px;border-top:1px solid #eee;"
                              "border-bottom:1px solid #eee;'><b style='font-size:12px;color:#444;'>"
                              "Data availability: </b>" + "".join(parts) + "</div>")

def _apply_structure_gating():
    """Disable only the auto-fetch controls when their source is unavailable.
    Manual PDB-ID entry and file upload always stay enabled as fallbacks."""
    av = state.get("avail", {})
    has_af = av.get("af") is not False   # unknown (None) -> keep enabled
    has_pdb = (av.get("pdb") or 0) > 0
    # AlphaFold fetch/download buttons
    _set_enabled([btn_fetch_af, btn_dl_af], has_af)
    # UniProt PDB dropdowns across tabs
    _set_enabled([b2b_pdb_dropdown, b2b_chain_dropdown,
                  rin_pdb_dd, rin_chain_dd,
                  tm_pdb1_dd, tm_pdb2_dd, tm_chain1_dd, tm_chain2_dd], has_pdb)

# ---- UI polish: button labels sized to content, consistent colors ----
def _fit_btn(b):
    try:
        b.layout.width = "max-content"
    except Exception:
        pass

# Bio2Byte 3D button: rename + green
btn_show_b2b3d.description = "Show 3D"
btn_show_b2b3d.button_style = "success"

# All Export buttons: blue
for _b in (btn_export_tab3, btn_export_b2b, btn_export_rin, btn_export_tm):
    _b.button_style = "primary"

# Size buttons to their labels so nothing is cut off
for _b in (btn_fetch_var, btn_dl_af,
           btn_fetch_seq, btn_run_b2b, btn_show_b2b, btn_show_b2b3d, btn_export_b2b,
           btn_range, btn_run, btn_export_tm):
    _fit_btn(_b)

tabs = w.Tab(children=[tab1, tab2, tab3, tab4, tab5, tab6])
titles = ["1) PTMs", "2) Variants", "3) 3D Viewer", "4) Bio2Byte", "5) RIN", "6) TM-align"]
for i, t in enumerate(titles):
    tabs.set_title(i, t)

display(w.VBox([top, protein_info_box, availability_box, tabs]))

